# Systematic Review Automation — Google Colab
**LLMs in Qualitative Data Analysis (2023–2026)**

Run cells top to bottom. Cell 5 gives you a public URL — no accounts needed.

---
### Before you start — add your OpenAI API key as a Colab Secret
1. Click the **🔑 key icon** in the left sidebar → **+ Add new secret**
2. Name: `OPENAI_API_KEY` · Value: your `sk-...` key · Toggle **Notebook access** ON

Your key is encrypted in your Google account — never visible in output, never sent anywhere.


## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/qbzdm2024/Zhihong.github.io.git'
BRANCH   = 'claude/systematic-review-automation-oC3BZ'
REPO_DIR = '/content/sysrev'
APP_DIR  = f'{REPO_DIR}/systematic-review'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo already present — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {APP_DIR}
print('Working directory:', os.getcwd())

In [ ]:
%%capture
!pip install \
    "fastapi==0.115.0" "uvicorn[standard]" \
    "pydantic>=2.7.0" "pydantic-settings>=2.3.0" \
    "openai>=1.50.0" "rispy>=0.9.0" \
    "python-multipart" "python-dotenv" "PyMuPDF" \
    "requests" "xmltodict" "playwright"

# Install Chromium browser for Playwright (used by the fulltext downloader
# to access Google Scholar and other JS-rendered pages like a real browser)
!playwright install chromium --with-deps 2>&1 | tail -5

In [ ]:
import fastapi, openai, pydantic, requests
try:
    import rispy
    rispy_ver = rispy.__version__
except ImportError:
    rispy_ver = 'NOT INSTALLED — re-run the pip cell above'

print(f'fastapi {fastapi.__version__} | openai {openai.__version__} | pydantic {pydantic.__version__}')
print(f'rispy   {rispy_ver}')

for d in ['data/raw','data/deduped','data/screened','data/extracted','data/output','data/pdfs']:
    os.makedirs(d, exist_ok=True)
print('Data directories ready.')

## Cell 2 — Load API key from Colab Secrets

In [ ]:
from google.colab import userdata

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    raise RuntimeError(
        'OPENAI_API_KEY not found in Colab Secrets.\n'
        'Click the 🔑 key icon → Add new secret → Name: OPENAI_API_KEY'
    )

if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith('sk-'):
    raise ValueError('Key must start with sk-')

with open('.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')
    f.write('MODEL_TITLE_SCREENING=gpt-4o-mini\n')
    f.write('MODEL_FULLTEXT_SCREENING=gpt-5-mini\n')
    f.write('MODEL_AGENT2_SCREENING=gpt-5\n')
    f.write('MODEL_EXTRACTION=gpt-5\n')
    f.write('MODEL_QA_ASSESSMENT=gpt-4o\n')
    f.write('MODEL_AGENT2_EXTRACTION=gpt-5.4-mini\n')
    f.write('CONFIDENCE_THRESHOLD=0.80\n')

print(f'✓ API key loaded (sk-...{OPENAI_API_KEY[-4:]})')
print('✓ .env written')
print()
print('Models:')
print('  Title screening   : gpt-4o-mini  (agent 1)  |  gpt-5      (agent 2)')
print('  Fulltext screening: gpt-5-mini   (agent 1)  |  gpt-5      (agent 2)')
print('  Extraction        : gpt-5        (agent 1)  |  gpt-5.4-mini (agent 2)')

## Cell 3a — Automated database search (free, no login required)

Searches **PubMed, arXiv, OpenAlex, and Semantic Scholar** automatically.
All results are saved to `data/raw/` ready for import.

These four sources are free and open — **no institutional access needed.**
For Scopus, Web of Science, PsycINFO, ACM, and IEEE, see Cell 3b below.

In [ ]:
import requests, json, time, xml.etree.ElementTree as ET
from urllib.parse import quote_plus
from datetime import datetime

os.makedirs('data/raw', exist_ok=True)

# ── Search configuration ────────────────────────────────
DATE_FROM = '2023'
DATE_TO   = '2026'
MAX_RESULTS_PER_SOURCE = 500   # increase if you want more

# ── Core search terms (from refined protocol) ───────────
LLM_TERMS = [
    'large language model', 'large language models', 'LLM', 'LLMs',
    'generative AI', 'generative artificial intelligence',
    'GPT-4', 'GPT-3', 'GPT-3.5', 'ChatGPT', 'GPT',
    'Claude', 'Llama', 'Llama 2', 'Llama 3',
    'Gemini', 'Mistral', 'PaLM', 'foundation model',
]
QDA_TERMS = [
    'qualitative analysis', 'qualitative research', 'qualitative data analysis',
    'thematic analysis', 'content analysis', 'grounded theory',
    'framework analysis', 'narrative analysis', 'discourse analysis',
    'inductive coding', 'deductive coding', 'open coding',
    'codebook', 'qualitative coding', 'interpretive phenomenological',
]

print(f'Search scope: {DATE_FROM}–{DATE_TO} | max {MAX_RESULTS_PER_SOURCE} per source')
print(f'LLM terms: {len(LLM_TERMS)} | QDA terms: {len(QDA_TERMS)}')

In [ ]:
# ════════════════════════════════════════════════════════
# PubMed  (NCBI E-utilities — completely free, no key)
# Uses your refined Title/Abstract-restricted strategy
# ════════════════════════════════════════════════════════
print('Searching PubMed with refined strategy...')

EUTILS = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

# ── Refined strategy: Title/Abstract field restrictions ──────────────────
# Matches the strategy you manually executed (saved in docs/search_strategies.md)
LLM_TIAB = (
    '"large language model"[Title/Abstract] OR "large language models"[Title/Abstract] '
    'OR ChatGPT[Title/Abstract] OR GPT-4[Title/Abstract] OR "GPT 4"[Title/Abstract] '
    'OR Claude[Title/Abstract] OR Llama[Title/Abstract] OR Gemini[Title/Abstract]'
)
QDA_TIAB = (
    '"qualitative analysis"[Title/Abstract] OR "qualitative data analysis"[Title/Abstract] '
    'OR "thematic analysis"[Title/Abstract] OR "content analysis"[Title/Abstract] '
    'OR "grounded theory"[Title/Abstract] OR coding[Title/Abstract] '
    'OR codebook[Title/Abstract] OR "theme development"[Title/Abstract] '
    'OR "automated coding"[Title/Abstract]'
)

pubmed_query = (
    f'({LLM_TIAB}) AND ({QDA_TIAB}) '
    f'AND ("{DATE_FROM}/01/01"[Date - Publication] : "{DATE_TO}/12/31"[Date - Publication]) '
    f'AND (english[Language])'
)

print(f'  Query length: {len(pubmed_query)} chars')

# Step 1: get IDs
search_r = requests.get(f'{EUTILS}/esearch.fcgi', params={
    'db': 'pubmed', 'term': pubmed_query,
    'retmax': MAX_RESULTS_PER_SOURCE, 'retmode': 'json',
    'usehistory': 'y',
}, timeout=30)
search_data = search_r.json()
count  = int(search_data['esearchresult']['count'])
webenv = search_data['esearchresult'].get('webenv', '')
qkey   = search_data['esearchresult'].get('querykey', '')
ids    = search_data['esearchresult']['idlist']
print(f'  Total matching: {count} | Fetching: {len(ids)}')

# Step 2: fetch XML records
if ids:
    time.sleep(0.4)  # NCBI rate limit: 3 req/s
    fetch_r = requests.get(f'{EUTILS}/efetch.fcgi', params={
        'db': 'pubmed', 'query_key': qkey, 'WebEnv': webenv,
        'rettype': 'xml', 'retmode': 'xml',
        'retmax': MAX_RESULTS_PER_SOURCE,
    }, timeout=60)
    out_path = 'data/raw/pubmed_results.xml'
    with open(out_path, 'wb') as f:
        f.write(fetch_r.content)
    print(f'  ✓ Saved {len(ids)} records → {out_path}')
    print(f'  Note: If you manually exported from PubMed, your .nbib file takes precedence')
    print(f'        (both will be imported; duplicates are removed automatically)')
else:
    print('  No PubMed results found.')

time.sleep(0.4)

In [ ]:
# ════════════════════════════════════════════════════════
# arXiv  (free API, no key — cs.AI, cs.CL, cs.HC)
# ════════════════════════════════════════════════════════
print('Searching arXiv...')

import xml.etree.ElementTree as ET

llm_arxiv = ' OR '.join(f'abs:"{t}"' for t in LLM_TERMS[:8])  # keep query short
qda_arxiv = ' OR '.join(f'abs:"{t}"' for t in QDA_TERMS[:8])
arxiv_query = f'({llm_arxiv}) AND ({qda_arxiv})'

arxiv_records = []
batch_size = 100
start = 0
import uuid

while start < MAX_RESULTS_PER_SOURCE:
    r = requests.get('http://export.arxiv.org/api/query', params={
        'search_query': arxiv_query,
        'start': start,
        'max_results': batch_size,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending',
    }, timeout=30)

    ns = {'atom': 'http://www.w3.org/2005/Atom'}
    root = ET.fromstring(r.content)
    entries = root.findall('atom:entry', ns)
    if not entries:
        break

    for entry in entries:
        published = entry.findtext('atom:published', '', ns)[:4]
        if published < DATE_FROM or published > DATE_TO:
            continue
        arxiv_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'arXiv',
            'title': (entry.findtext('atom:title', '', ns) or '').strip().replace('\n', ' '),
            'authors': '; '.join(
                (a.findtext('atom:name', '', ns) or '')
                for a in entry.findall('atom:author', ns)
            ),
            'year': int(published) if published.isdigit() else None,
            'journal_venue': 'arXiv preprint',
            'abstract': (entry.findtext('atom:summary', '', ns) or '').strip().replace('\n', ' '),
            'doi': '',
            'url': entry.findtext('atom:id', '', ns),
        })

    start += batch_size
    if len(entries) < batch_size:
        break
    time.sleep(3)  # arXiv asks for 3s between requests

if arxiv_records:
    out_path = 'data/raw/arxiv_results.json'
    with open(out_path, 'w') as f:
        json.dump(arxiv_records, f, indent=2)
    print(f'  ✓ Saved {len(arxiv_records)} records → {out_path}')
else:
    print('  No arXiv results in date range.')

In [ ]:
# ════════════════════════════════════════════════════════
# OpenAlex  (free, open — covers Scopus/WoS/PubMed)
# https://openalex.org — no API key needed for <100k req/day
# ════════════════════════════════════════════════════════
print('Searching OpenAlex...')

# Build search query — OpenAlex uses simple keyword search
llm_kw  = ' OR '.join(f'"{t}"' for t in [
    'large language model', 'ChatGPT', 'GPT-4', 'LLM', 'Llama', 'Gemini', 'generative AI'
])
qda_kw  = ' OR '.join(f'"{t}"' for t in [
    'qualitative analysis', 'thematic analysis', 'content analysis',
    'grounded theory', 'qualitative research', 'qualitative coding',
])

openalex_records = []
page = 1
per_page = 100
EMAIL = 'sysrev@example.com'  # polite pool (faster responses)

while len(openalex_records) < MAX_RESULTS_PER_SOURCE:
    r = requests.get('https://api.openalex.org/works', params={
        'search': f'({llm_kw}) AND ({qda_kw})',
        'filter': f'publication_year:{DATE_FROM}-{DATE_TO},language:en',
        'per-page': per_page,
        'page': page,
        'mailto': EMAIL,
    }, timeout=30)

    if r.status_code != 200:
        print(f'  OpenAlex error {r.status_code}: {r.text[:200]}')
        break

    data = r.json()
    results = data.get('results', [])
    if not results:
        break

    for w in results:
        authors = '; '.join(
            a.get('author', {}).get('display_name', '')
            for a in (w.get('authorships') or [])[:6]
        )
        abstract = ''
        if w.get('abstract_inverted_index'):
            # Reconstruct abstract from inverted index
            idx = w['abstract_inverted_index']
            words = {pos: word for word, positions in idx.items() for pos in positions}
            abstract = ' '.join(words[i] for i in sorted(words))

        doi = w.get('doi') or ''
        if doi.startswith('https://doi.org/'):
            doi = doi[len('https://doi.org/'):]

        venue = ''
        if w.get('primary_location') and w['primary_location'].get('source'):
            venue = w['primary_location']['source'].get('display_name', '')

        openalex_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'OpenAlex',
            'title': w.get('title') or '',
            'authors': authors,
            'year': w.get('publication_year'),
            'journal_venue': venue,
            'abstract': abstract,
            'doi': doi,
            'url': w.get('id') or '',
        })

    total = data.get('meta', {}).get('count', 0)
    print(f'  Page {page}: {len(results)} results (total available: {total})')

    if page * per_page >= min(total, MAX_RESULTS_PER_SOURCE):
        break
    page += 1
    time.sleep(1)

if openalex_records:
    out_path = 'data/raw/openalex_results.json'
    with open(out_path, 'w') as f:
        json.dump(openalex_records, f, indent=2)
    print(f'  ✓ Saved {len(openalex_records)} records → {out_path}')
else:
    print('  No OpenAlex results.')

In [ ]:
# ════════════════════════════════════════════════════════
# Semantic Scholar  (free API — strong CS/AI coverage)
# https://api.semanticscholar.org
# ════════════════════════════════════════════════════════
print('Searching Semantic Scholar...')

s2_records = []
offset = 0
limit = 100
query = 'large language model qualitative analysis thematic analysis coding'

while len(s2_records) < MAX_RESULTS_PER_SOURCE:
    r = requests.get('https://api.semanticscholar.org/graph/v1/paper/search', params={
        'query': query,
        'fields': 'title,authors,year,abstract,externalIds,venue,publicationDate',
        'limit': limit,
        'offset': offset,
    }, timeout=30)

    if r.status_code == 429:
        print('  Rate limited — waiting 10s...')
        time.sleep(10)
        continue

    if r.status_code != 200:
        print(f'  S2 error {r.status_code}')
        break

    data = r.json()
    papers = data.get('data', [])
    if not papers:
        break

    for p in papers:
        year = p.get('year')
        if year and (year < int(DATE_FROM) or year > int(DATE_TO)):
            continue

        doi = (p.get('externalIds') or {}).get('DOI', '')
        authors = '; '.join(
            a.get('name', '') for a in (p.get('authors') or [])[:6]
        )
        s2_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'SemanticScholar',
            'title': p.get('title') or '',
            'authors': authors,
            'year': year,
            'journal_venue': p.get('venue') or '',
            'abstract': p.get('abstract') or '',
            'doi': doi,
            'url': f'https://www.semanticscholar.org/paper/{p["paperId"]}',
        })

    total = data.get('total', 0)
    print(f'  Offset {offset}: got {len(papers)} (total: {total})')

    if offset + limit >= min(total, MAX_RESULTS_PER_SOURCE):
        break
    offset += limit
    time.sleep(1.5)

if s2_records:
    out_path = 'data/raw/semanticscholar_results.json'
    with open(out_path, 'w') as f:
        json.dump(s2_records, f, indent=2)
    print(f'  ✓ Saved {len(s2_records)} records → {out_path}')

print('\n── Automated search summary ──────────────────────')
for fname in ['pubmed_results.xml','arxiv_results.json',
              'openalex_results.json','semanticscholar_results.json']:
    p = f'data/raw/{fname}'
    if os.path.exists(p):
        size = os.path.getsize(p)
        print(f'  ✓ {fname:45s} {size:>8,} bytes')
    else:
        print(f'  ✗ {fname} — not generated')

## Cell 3b — Manual exports (institutional databases)

The databases below require university/library login. Run these searches yourself,
then upload the exported files using the upload widget below.

---

### PubMed (manual export — NBIB format)
1. Run your search at [pubmed.ncbi.nlm.nih.gov](https://pubmed.ncbi.nlm.nih.gov)
2. Use the refined strategy saved in `docs/search_strategies.md`
3. Select all results → **Send to** → **Citation manager** → **Create file**
4. Save as `pubmed.nbib`  *(the pipeline reads NBIB natively)*

**Refined PubMed strategy:**
```
(
  ("large language model"[Title/Abstract] OR "large language models"[Title/Abstract]
   OR ChatGPT[Title/Abstract] OR GPT-4[Title/Abstract] OR "GPT 4"[Title/Abstract]
   OR Claude[Title/Abstract] OR Llama[Title/Abstract] OR Gemini[Title/Abstract])
  AND
  ("qualitative analysis"[Title/Abstract] OR "qualitative data analysis"[Title/Abstract]
   OR "thematic analysis"[Title/Abstract] OR "content analysis"[Title/Abstract]
   OR "grounded theory"[Title/Abstract] OR coding[Title/Abstract]
   OR codebook[Title/Abstract] OR "theme development"[Title/Abstract]
   OR "automated coding"[Title/Abstract])
)
AND ("2023/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])
AND (english[Language])
```

---

### Scopus
1. Go to [scopus.com](https://www.scopus.com) (requires institutional login)
2. Click **Advanced search** → paste strategy from `docs/search_strategies.md`
3. Filter: **Date 2023–2026** · **Language: English**
4. Select all → **Export** → **RIS** → include Abstract + Keywords
5. Save as `scopus.ris`

### Web of Science
1. Go to [webofscience.com](https://www.webofscience.com) → **Advanced Search**
2. Paste strategy, set **Timespan: 2023–2026**, **Language: English**
3. Select all → **Export** → **RIS** → include all fields
4. Save as `wos.ris`

### PsycINFO (via EBSCOhost)
1. Access via your library → EBSCOhost → PsycINFO database
2. **Advanced Search** → paste query
3. Limiter: **Publication Date 2023–2026** · **Language: English** · **Peer Reviewed**
4. Select all → **Export** → **Direct Export to RIS**
5. Save as `psycinfo.ris`

### ACM Digital Library
1. Go to [dl.acm.org](https://dl.acm.org) → **Advanced Search**
2. Filter: **Published 2023–2026**
3. Export → **BibTeX** or **CSV**
4. Save as `acm.bib` or `acm.csv`

### IEEE Xplore
1. Go to [ieeexplore.ieee.org](https://ieeexplore.ieee.org) → **Advanced Search**
2. Paste query, set **Year: 2023–2026**
3. Export → **CSV** → include Abstract
4. Save as `ieee.csv`


### Upload manual export files

In [ ]:
from google.colab import files as colab_files

SUPPORTED = ('.nbib', '.ris', '.csv', '.bib', '.json', '.xml', '.txt')
print('Upload your exported files:')
print('  • PubMed  → pubmed.nbib  (Citation manager export)')
print('  • Scopus  → scopus.ris')
print('  • WoS     → wos.ris')
print('  • Others  → psycinfo.ris, acm.bib, ieee.csv, ...')
print('\nPress Cancel if you have no files to upload right now.\n')

try:
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        if any(fname.lower().endswith(ext) for ext in SUPPORTED):
            dest = f'data/raw/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            fmt = fname.rsplit('.', 1)[-1].upper()
            print(f'  ✓ {fname} ({len(content):,} bytes) → data/raw/  [{fmt}]')
        else:
            print(f'  ✗ Skipped (unsupported format): {fname}')
except Exception:
    print('No files uploaded — you can re-run this cell later.')

print('\nAll files in data/raw/:')
!ls -lh data/raw/ 2>/dev/null || echo '  (empty)'

## Cell 4 — Bulk PDF upload (97+ files at once)

Uploading many PDFs one at a time is slow and error-prone in Colab.
The fastest method is to **copy your PDFs directly into your Google Drive** and
let this cell import them all in one go.

### Step-by-step

1. **Rename** each PDF to its `upload_filename_hint` value (the UUID from
   `manual_fulltext_needed.csv`).  Example: `a1b2c3d4-5678-abcd-ef01-234567890abc.pdf`
   You can also use the sanitised DOI (`10.1016_j.foo.2024.101.pdf`) if renaming
   by UUID is inconvenient.

2. **Upload to Drive** — open [drive.google.com](https://drive.google.com) in
   your browser, navigate to the folder
   **`My Drive → SysRev_LLMs_QDA → pdfs`** and drag-and-drop all PDFs at once.
   (Create the `pdfs` sub-folder if it doesn't exist yet.)

3. **Run this cell** — it copies every PDF from that Drive folder into
   `data/pdfs/` and registers them with the server.

4. Then run Cell 6 → `run_stage("fulltext_screening")`.

---

> **Alternative (small batches):** if you only have a few files you can still
> use the Colab file-picker below — hold **Ctrl / ⌘** to select multiple files.

In [ ]:
import shutil, os, requests
from pathlib import Path

PORT     = 8000
BASE     = f'http://localhost:{PORT}/api'
PDF_DIR  = '/content/sysrev/systematic-review/data/pdfs'
DRIVE_PDF_DIR = '/content/drive/MyDrive/SysRev_LLMs_QDA/pdfs'

os.makedirs(PDF_DIR, exist_ok=True)

# ── Method A: bulk import from Google Drive ───────────────────────────────
if os.path.isdir(DRIVE_PDF_DIR):
    drive_pdfs = [f for f in Path(DRIVE_PDF_DIR).iterdir()
                  if f.suffix.lower() in ('.pdf', '.txt')]
    print(f'Found {len(drive_pdfs)} PDF/TXT files in Drive folder: {DRIVE_PDF_DIR}')

    copied = 0
    for src in drive_pdfs:
        dest = Path(PDF_DIR) / src.name
        if not dest.exists():
            shutil.copy2(src, dest)
            copied += 1

    print(f'Copied {copied} new files → data/pdfs/  '
          f'({len(drive_pdfs) - copied} already present, skipped)')

    # Register all files in data/pdfs/ with the running server
    try:
        r = requests.post(f'{BASE}/pdfs/register-from-disk', timeout=60)
        result = r.json()
        print(f'\nServer registration:')
        print(f'  Matched to records : {result["registered"]}')
        print(f'  Unmatched (no record found) : {result["unmatched"]}')
        if result['unmatched']:
            print('  Unmatched files (check filenames match upload_filename_hint):')
            for row in result['results']:
                if row['status'] == 'unmatched':
                    print(f'    {row["file"]}')
        print('\nNext: run Cell 6 → run_stage("fulltext_screening")')
    except Exception as e:
        print(f'Server not running ({e}) — start Cell 5 first, then re-run this cell.')

else:
    print(f'Drive folder not found: {DRIVE_PDF_DIR}')
    print('→ Create "pdfs" sub-folder inside My Drive/SysRev_LLMs_QDA/ on drive.google.com')
    print('  and upload your PDFs there, then re-run this cell.')
    print()
    print('─── Alternative: Colab file-picker (small batches only) ────────────────')
    from google.colab import files as colab_files
    try:
        uploaded = colab_files.upload()   # Ctrl/⌘-click to select multiple
        saved = 0
        for fname, content in uploaded.items():
            if fname.lower().endswith(('.pdf', '.txt')):
                with open(f'{PDF_DIR}/{fname}', 'wb') as f:
                    f.write(content)
                saved += 1
                print(f'  ✓ {fname}')
        if saved:
            r = requests.post(f'{BASE}/pdfs/register-from-disk', timeout=60)
            result = r.json()
            print(f'\nRegistered: {result["registered"]}  |  Unmatched: {result["unmatched"]}')
    except Exception:
        print('Upload cancelled.')

## Cell 5 — Start server + open public URL

Uses Cloudflare Tunnel — no account, no token, free.

In [ ]:
import subprocess, time, re, sys, os, urllib.request
from IPython.display import display, HTML

APP_DIR = '/content/sysrev/systematic-review'
PORT = 8000

os.system(f'fuser -k {PORT}/tcp 2>/dev/null; pkill cloudflared 2>/dev/null; true')
time.sleep(1)

# ── Mount Drive + enable per-checkpoint sync ─────────────────────────────
from google.colab import drive as _drive
_drive.mount('/content/drive', force_remount=False)

DRIVE_DIR  = '/content/drive/MyDrive/SysRev_LLMs_QDA'
DRIVE_FILE = f'{DRIVE_DIR}/pipeline_state.jsonl'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ['DRIVE_STATE_FILE'] = DRIVE_FILE
print(f'✓ Drive sync active — checkpoints save to:\n  {DRIVE_FILE}')

if not os.path.exists('/content/cloudflared'):
    print('Downloading cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
         -O /content/cloudflared && chmod +x /content/cloudflared

import tempfile, io
_stderr_log = tempfile.NamedTemporaryFile(mode="w", suffix=".log", delete=False)
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.main:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'info'],
    cwd=APP_DIR,
    env=os.environ.copy(),
    stdout=_stderr_log, stderr=_stderr_log,
)

print('Starting server', end='')
_started = False
for _ in range(40):
    time.sleep(1)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/api/pipeline/status', timeout=2)
        print(' ready.')
        _started = True
        break
    except Exception:
        print('.', end='', flush=True)
if not _started:
    _stderr_log.flush()
    with open(_stderr_log.name) as _f:
        _err = _f.read().strip()
    print('\n')
    print('ERROR: Server did not start. Last log output:')
    print(_err[-3000:] if _err else '(no output captured)')
    print('\nFix the error above, then re-run this cell.')
    raise SystemExit('Server failed to start')

tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

public_url = None
print('Opening Cloudflare tunnel', end='')
for _ in range(40):
    line = tunnel.stdout.readline().decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print(' done.')
        break
    print('.', end='', flush=True)

if public_url:
    app_url = f'{public_url}/app'
    print(f'\n  UI:  {app_url}')
    print(f'  API: {public_url}/docs')
    display(HTML(f'''
    <div style="margin:12px 0;padding:18px;background:#0f1117;border:2px solid #6c8eff;
      border-radius:10px;font-family:monospace;">
      <div style="color:#8890aa;font-size:11px;margin-bottom:6px;">OPEN IN YOUR BROWSER</div>
      <a href="{app_url}" target="_blank" rel="noopener"
        style="color:#6c8eff;font-size:18px;font-weight:bold;text-decoration:none;">{app_url}</a>
      <div style="color:#4caf81;font-size:12px;margin-top:10px;">
        ✓ API key from Colab Secrets &nbsp;|&nbsp; ✓ No account needed &nbsp;|&nbsp; ✓ HTTPS
      </div>
    </div>
    '''))
else:
    print('\nERROR: Could not get tunnel URL. Re-run this cell.')

## Cell 6 — Run pipeline stages

Run from here, or use the **Run Pipeline** panel in the UI.

In [ ]:
import requests, time

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

def _server_ok():
    try:
        requests.get(f'{BASE}/pipeline/status', timeout=3)
        return True
    except Exception:
        print('╔══════════════════════════════════════════════════════════╗')
        print('║  SERVER NOT RUNNING — please run Cell 5 first, then      ║')
        print('║  re-run this cell and try again.                         ║')
        print('╚══════════════════════════════════════════════════════════╝')
        return False

def run_stage(stage, limit=None, force=False):
    """Run a pipeline stage.
    import / dedup / reset_screening / reset_failed_screenings / reset_fulltext_screening → synchronous.
    title_screening / fulltext_screening / extraction → background task;
                      call status() to monitor progress.
    Use force=True with reset_fulltext_screening to also reset bulk-excluded records.
    """
    if not _server_ok(): return
    r = requests.post(f'{BASE}/pipeline/run', json={'stage': stage, 'limit': limit, 'force': force}, timeout=300)
    r.raise_for_status()
    resp = r.json()

    if resp.get('status') == 'completed':
        print(f'✓ {stage} complete.')
        if 'stats' in resp:
            st = resp['stats']
            if stage == 'import':
                print(f'  Imported: {st.get("imported","?")} records')
                for fname, n in (st.get('per_file') or {}).items():
                    print(f'    {str(n):>5}  {fname}')
            elif stage == 'dedup':
                print(f'  Input: {st.get("total_input","?")}  |  Unique: {st.get("unique_records","?")}')
                print(f'  DOI dupes: {st.get("duplicates_by_doi","?")}  '
                      f'Title dupes: {st.get("duplicates_by_exact_title","?")}  '
                      f'Fuzzy dupes: {st.get("duplicates_by_fuzzy_title","?")}')
            elif stage == 'reset_screening':
                print(f'  Rolled back: {st.get("rolled_back","?")} records → DEDUP stage')
                print(f'  Human-verified decisions preserved: {st.get("human_verified_kept","?")}')
            elif stage == 'reset_failed_screenings':
                print(f'  Reset error-path records: {st.get("rolled_back","?")} records → DEDUP stage')
                print(f'  (Records with valid INCLUDE/EXCLUDE results were NOT touched)')
            elif stage == 'restore_bulk_excluded':
                print(f'  Restored: {st.get("restored","?")} records → Needs Human Verification')
        status()
    else:
        print(f'⏳ {stage} started in background.')
        print('  Call status() to monitor. This can take minutes to hours.')

def _print_status(data):
    c = data['prisma_counts']
    log = data.get('stage_log', {})
    running = data.get('running_stage', '')
    warnings = data.get('warnings', [])

    if running:
        print(f'\n  ⏳  STAGE RUNNING: {running}  — call status() again in ~30s\n')

    for w in warnings:
        print(f'\n  {w}\n')

    print('── PRISMA Flow ───────────────────────────────────────────────')
    print(f'  Identified (all imports)            {c.get("identified",0):>6}')
    print(f'  Duplicates removed                  {c.get("duplicates_removed",0):>6}')
    print(f'  After dedup (unique)                {c.get("after_dedup",0):>6}')
    ats = c.get("awaiting_title_screening", 0)
    hint = '  ← run_stage("title_screening") next' if ats > 0 and not running else ''
    print(f'  Awaiting title screening            {ats:>6}{hint}')
    print(f'  Title/abstract excluded             {c.get("title_abstract_excluded",0):>6}')
    print(f'  Title/abstract included             {c.get("title_abstract_included",0):>6}')
    print(f'  Full text needed (upload PDFs)      {c.get("full_text_needed",0):>6}')
    print(f'  Full text excluded                  {c.get("full_text_excluded",0):>6}')
    print(f'  Final included                      {c.get("final_included",0):>6}')
    # Show per-stage human verification counts separately
    nhv_title    = c.get('needs_human_title_screening', 0)
    nhv_fulltext = c.get('needs_human_fulltext_screening', 0)
    nhv_extract  = c.get('needs_human_extraction', 0)
    nhv_total    = c.get('needs_human_verification', 0)
    if nhv_title > 0:
        print(f'  Needs review — title screening      {nhv_title:>6}  ⚠ ← review in UI')
    if nhv_fulltext > 0:
        print(f'  Needs review — fulltext screening   {nhv_fulltext:>6}  ⚠ ← review in UI')
    if nhv_extract > 0:
        print(f'  Needs review — data extraction      {nhv_extract:>6}  ⚠ ← review in UI')
    if nhv_total == 0:
        print(f'  Needs human verification            {0:>6}')
    print()

def status():
    if not _server_ok(): return
    _print_status(requests.get(f'{BASE}/pipeline/status').json())

def uncertain():
    if not _server_ok(): return
    data = requests.get(f'{BASE}/records/uncertain/list').json()
    print(f'\nNeeds Human Verification: {data["count"]} records')
    for r in data['records'][:10]:
        print(f'  [{r["record_id"][:8]}] {r["title"][:70]}')

def fulltext_needed():
    if not _server_ok(): return
    data = requests.get(f'{BASE}/records/fulltext-needed/list').json()
    print(f'\nFull Text Needed: {data["count"]} papers')
    for r in data['records'][:10]:
        print(f'  {r["doi"] or "no-doi":40s}  {r["title"][:50]}')

print('Helpers ready.')
print('  run_stage("import")                  — import files (instant)')
print('  run_stage("dedup")                   — deduplicate (instant)')
print('  run_stage("reset_failed_screenings") — reset only error-path records back to DEDUP')
print('  run_stage("reset_screening")         — roll back ALL non-verified screening results')
print('  run_stage("restore_bulk_excluded")   — restore records accidentally bulk-excluded')
print('  run_stage("title_screening")         — start AI screening (background)')
print('  status()                             — PRISMA flow counts + warnings')

### Cell 6a — Diagnostics: check what files are in data/raw

Run this before import to verify all your files are present and readable.
If the count is lower than expected, see the encoding fix notes below.

In [ ]:
import os, sys, subprocess

# ── Ensure rispy is available (safe to run even if already installed) ────
try:
    import rispy
except ImportError:
    print('Installing rispy...', end='')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'rispy', '-q'], check=True)
    import rispy
    print(' done.')

RAW = 'data/raw'
SUPPORTED = {'.nbib', '.ris', '.csv', '.bib', '.json', '.xml', '.txt'}

if not os.path.exists(RAW):
    print('data/raw does not exist — run Cell 1 first (or Cell 8 to restore from Drive).')
else:
    files = sorted(f for f in os.listdir(RAW) if os.path.splitext(f)[1].lower() in SUPPORTED)
    other = sorted(f for f in os.listdir(RAW) if os.path.splitext(f)[1].lower() not in SUPPORTED and not f.startswith('.'))

    if not files and not other:
        print('data/raw/ is EMPTY.')
        print('→ Run Cell 3a for automated searches, or Cell 3b to upload your export files.')
    else:
        print(f'Supported files in data/raw/ ({len(files)} files):')
        total_bytes = 0
        for fname in files:
            size = os.path.getsize(f'{RAW}/{fname}')
            total_bytes += size
            ext = os.path.splitext(fname)[1].upper()
            print(f'  {ext:6s}  {size:>10,} bytes  {fname}')
        print(f'\n  Total: {total_bytes:,} bytes across {len(files)} files')
        if other:
            print(f'\nUnsupported files (will be skipped): {other}')

    # Quick parse-test: count records per file before import
    print('\n── Quick record count (before import/dedup) ────────────────')
    sys.path.insert(0, '.')
    try:
        # Reload importer to pick up rispy now that it's installed
        import importlib
        if 'pipeline.importer' in sys.modules:
            importlib.reload(sys.modules['pipeline.importer'])
        from pipeline.importer import (
            import_ris, import_nbib, import_csv, import_pubmed_xml, import_json
        )
        parsers = {
            '.nbib': import_nbib, '.txt': import_nbib,
            '.ris':  lambda p: import_ris(p, source_db='check'),
            '.csv':  lambda p: import_csv(p, source_db='check'),
            '.xml':  import_pubmed_xml,
            '.json': lambda p: import_json(p, source_db='check'),
        }
        grand_total = 0
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            path = f'{RAW}/{fname}'
            if ext in parsers:
                try:
                    recs = parsers[ext](path)
                    n = len(recs)
                    grand_total += n
                    print(f'  {n:>5} records  {fname}')
                except Exception as e:
                    print(f'  ERROR  {fname}: {e}')
        print(f'\n  Grand total (before dedup): {grand_total} records')
        if grand_total > 0:
            print('\n  ✓ Files look good — run Cell 6 → run_stage("import") to import them.')
    except ImportError as e:
        print(f'  Could not import parsers: {e}')
        print('  Make sure Cell 1 has been run in this session.')

### Cell 6b — Debug: verify API key + check agent errors

Run this if title screening is stuck or all records show "Needs Human Verification".

In [ ]:
import requests

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

# ── Step 1: Test that your API key works ─────────────────────────────────
print('Testing OpenAI API key...')
try:
    r = requests.get(f'{BASE}/debug/test-api-key', timeout=30)
    result = r.json()
    if result.get('status') == 'ok':
        print(f'  ✓ API key OK — model {result["model"]} responded:')
        print(f'    {result["response"]}')
    else:
        print(f'  ✗ API ERROR: {result["error_type"]}: {result["error_message"]}')
        print(f'    Hint: {result.get("hint", "")}')
        if 'Authentication' in result.get('error_type', '') or 'Invalid' in result.get('error_message', ''):
            print()
            print('  ── How to fix ─────────────────────────────────────────')
            print('  1. Go to Cell 2 and verify your OPENAI_API_KEY secret')
            print('  2. Re-run Cell 2 to rewrite .env')
            print('  3. Re-run Cell 5 to restart the server')
            print('  4. Re-run this cell to verify the fix')
except Exception as e:
    print(f'  Cannot reach server: {e}')
    print('  → Run Cell 5 first')

print()

# ── Step 2: Show recent agent screening errors ────────────────────────────
print('Recent agent errors (last 20):')
try:
    r2 = requests.get(f'{BASE}/debug/agent-errors', timeout=10)
    err_data = r2.json()
    if err_data['error_count'] == 0:
        print('  (none — agents running without errors)')
    else:
        print(f'  Total: {err_data["error_count"]} errors')
        for msg in err_data['errors'][-20:]:
            print(f'  {msg}')
except Exception as e:
    print(f'  Cannot reach server: {e}')

### Cell 6c — Auto-save to Drive during long runs

Run this **before** starting title_screening. It saves `pipeline_state.jsonl`
to Google Drive every 10 minutes in the background so a Colab timeout won't
lose your screening progress.

In [ ]:
import os
from pathlib import Path

# Cell 5 already sets DRIVE_STATE_FILE and passes it to the server process.
# Every 50 records the pipeline writes pipeline_state.jsonl and immediately
# copies it to Drive — no extra thread needed.

drive_file = os.environ.get('DRIVE_STATE_FILE', '')
local_file = '/content/sysrev/systematic-review/data/pipeline_state.jsonl'

if drive_file:
    print(f'✓ Drive sync is ACTIVE (set by Cell 5)')
    print(f'  Checkpoints every 50 records → {drive_file}')
    if Path(drive_file).exists():
        size = Path(drive_file).stat().st_size
        lines = sum(1 for _ in open(drive_file))
        print(f'  Current Drive file: {lines} records  ({size:,} bytes)')
    else:
        print('  Drive file not yet created (will appear after first checkpoint)')
else:
    print('⚠️  DRIVE_STATE_FILE not set — re-run Cell 5 to enable Drive sync.')
    print('   Without this, a Colab disconnect will lose all screening progress.')

print()
print('If Colab resets mid-screening:')
print('  Cell 1 → Cell 2 → Cell 8 (restore) → Cell 5 → Cell 6')
print('  run_stage("title_screening") will resume from the last checkpoint.')

In [ ]:
# Uncomment and run one at a time. Check status() between stages.

# run_stage('import')          # load all files from data/raw/
# status()

# run_stage('dedup')           # remove duplicates
# status()

# run_stage('title_screening') # two-agent title/abstract screening
# status()
# uncertain()                  # ← review these in the UI before continuing

# run_stage('fulltext_screening')
# fulltext_needed()            # ← upload those PDFs, then re-run

# run_stage('extraction')
# status()

## Cell 7 — Save to Google Drive (run before closing)

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)
DRIVE = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP   = '/content/sysrev/systematic-review'
os.makedirs(DRIVE, exist_ok=True)

for src, label in [
    (f'{APP}/data/pipeline_state.jsonl', 'pipeline_state.jsonl'),
    (f'{APP}/data/output', 'output/'),
    (f'{APP}/data/raw',    'raw/'),
    (f'{APP}/data/pdfs',   'pdfs/'),
]:
    if not os.path.exists(src):
        continue
    if os.path.isfile(src):
        shutil.copy(src, f'{DRIVE}/{label}')
    else:
        if os.listdir(src):
            shutil.copytree(src, f'{DRIVE}/{label.rstrip("/")}', dirs_exist_ok=True)
    print(f'  ✓ Saved: {label}')

print(f'\nSaved to Google Drive: {DRIVE}')

## Cell 8 — Restore after session reset

Run: **Cell 1 → Cell 2 → this cell → Cell 5** to resume.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)
DRIVE = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP   = '/content/sysrev/systematic-review'

if not os.path.exists(DRIVE):
    print('No saved session in Drive — starting fresh.')
else:
    restored = []
    for src_name, dest_path, is_file in [
        ('pipeline_state.jsonl', f'{APP}/data/pipeline_state.jsonl', True),
        ('output',               f'{APP}/data/output',               False),
        ('raw',                  f'{APP}/data/raw',                  False),
        ('pdfs',                 f'{APP}/data/pdfs',                 False),
    ]:
        src = f'{DRIVE}/{src_name}'
        if not os.path.exists(src):
            continue
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        if is_file:
            shutil.copy(src, dest_path)
        else:
            shutil.copytree(src, dest_path, dirs_exist_ok=True)
        restored.append(src_name)

    print(f'Restored: {" | ".join(restored) or "nothing found"}')
    print('Now run Cell 5 to restart the server.')

### Cell 8b — Reset old UNCERTAIN results + re-start screening

Run this **after Cell 5** when restoring from Drive.
It detects records that were screened under the old (broken) comparison logic,
resets them to DEDUP stage, then re-starts screening with the fixed logic.

**Order:** Cell 1 → Cell 2 → Cell 8 → Cell 5 → Cell 6 → **Cell 8b**

Human-verified decisions (if any) are always preserved.

In [ ]:
import requests, time

PORT = 8000
BASE = f"http://localhost:{PORT}/api"

def _server_ok():
    try:
        requests.get(f"{BASE}/pipeline/status", timeout=3)
        return True
    except Exception:
        print("Server not running — run Cell 5 first.")
        return False

if not _server_ok():
    raise SystemExit("Start server first (Cell 5), then re-run this cell.")

# ── Check current status ────────────────────────────────────────────────
st = requests.get(f"{BASE}/pipeline/status").json()
c  = st["prisma_counts"]
nhv         = c.get("needs_human_verification", 0)
after_dedup = c.get("after_dedup", 0)
awaiting    = c.get("awaiting_title_screening", 0)

print(f"Restored state:")
print(f"  After dedup (unique records)  : {after_dedup}")
print(f"  Awaiting title screening      : {awaiting}")
print(f"  Needs human verification      : {nhv}")
print()

if nhv > 0 and awaiting == 0:
    print(f"Detected {nhv} records from old screening run.")
    print("Resetting to DEDUP stage so they can be re-screened with fixed logic...")
    r = requests.post(f"{BASE}/pipeline/run",
                      json={"stage": "reset_screening"}, timeout=120)
    r.raise_for_status()
    resp = r.json()
    rolled = resp.get("stats", {}).get("rolled_back", "?")
    kept   = resp.get("stats", {}).get("human_verified_kept", 0)
    print(f"  ✓ Rolled back   : {rolled} records → DEDUP stage")
    print(f"  ✓ Preserved     : {kept} human-verified decisions (untouched)")
    print()
elif awaiting > 0:
    print(f"{awaiting} records already awaiting screening — no reset needed.")
else:
    print("State looks clean — nothing to reset.")

# ── Re-check status then launch screening ────────────────────────────────
st2  = requests.get(f"{BASE}/pipeline/status").json()
c2   = st2["prisma_counts"]
ready = c2.get("awaiting_title_screening", 0)

print(f"Ready to screen: {ready} records")

if ready > 0:
    print("Starting title screening with fixed comparison logic...")
    r2 = requests.post(f"{BASE}/pipeline/run",
                       json={"stage": "title_screening"}, timeout=60)
    r2.raise_for_status()
    print("⏳ Screening started in background.")
    print("   Call status() from Cell 6 to monitor progress.")
    print("   Expected: mix of Included / Excluded / some Needs Human Verification")
else:
    print("Nothing to screen. Check that Cell 8 (restore) ran successfully.")

### Cell 8c — Quick Resume after Colab crash / 401 errors

**Use this after any Colab disconnect or 401-error flood.**

Order: **Cell 1 → Cell 2 → Cell 8 → Cell 5 → Cell 6 → Cell 8c**

What it does:
- Counts records that errored mid-run (got 401 or other API errors — no stored decision)
- Resets *only* those records back to DEDUP stage (preserves all valid Include/Exclude results)
- Starts title screening immediately so they are retried with your fresh API key

In [ ]:
import requests

PORT = 8000
BASE = f"http://localhost:{PORT}/api"

def _server_ok():
    try:
        requests.get(f"{BASE}/pipeline/status", timeout=3)
        return True
    except Exception:
        print("Server not running — run Cell 5 first.")
        return False

if not _server_ok():
    raise SystemExit("Start server first (Cell 5), then re-run this cell.")

# ── Check current state ─────────────────────────────────────────────────
st = requests.get(f"{BASE}/pipeline/status").json()
c  = st["prisma_counts"]
after_dedup = c.get("after_dedup", 0)
awaiting    = c.get("awaiting_title_screening", 0)
included    = c.get("title_abstract_included", 0)
excluded    = c.get("title_abstract_excluded", 0)
nhv         = c.get("needs_human_verification", 0)

print("── Current state ──────────────────────────────────────────────")
print(f"  After dedup             : {after_dedup}")
print(f"  Already screened        : {included} included  |  {excluded} excluded")
print(f"  Needs human review      : {nhv}")
print(f"  Awaiting re-screening   : {awaiting}")
print()

# ── Reset only the error-path records (401s, timeouts, etc.) ────────────
# These have stage=TITLE_SCREENING + decision=UNCERTAIN + no stored result.
# Records with a valid INCLUDE/EXCLUDE result are NOT touched.
r = requests.post(f"{BASE}/pipeline/run",
                  json={"stage": "reset_failed_screenings"}, timeout=120)
r.raise_for_status()
resp   = r.json()
rolled = resp.get("stats", {}).get("rolled_back", 0)
print(f"Error-path records reset to DEDUP: {rolled}")
print("(Valid Include/Exclude results preserved)")
print()

# ── Re-check then launch ─────────────────────────────────────────────────
st2   = requests.get(f"{BASE}/pipeline/status").json()
ready = st2["prisma_counts"].get("awaiting_title_screening", 0)
print(f"Ready to screen: {ready} records")

if ready > 0:
    r2 = requests.post(f"{BASE}/pipeline/run",
                       json={"stage": "title_screening"}, timeout=60)
    r2.raise_for_status()
    print("Screening started in background.")
    print("Call status() from Cell 6 to monitor progress.")
elif rolled == 0:
    print("No error-path records found.")
    if nhv > 0:
        print(f"There are {nhv} records needing human review — use the UI or run Cell 8b")
        print("to reset ALL uncertain records if you want them re-screened.")
    else:
        print("All records already have valid decisions — nothing to retry.")

## Cell 8d — Full-Text Auto-Download (302 included records)

Automatically fetches open-access PDFs via **Unpaywall → Semantic Scholar → OpenAlex → Europe PMC**.

- Records where a PDF is found: kept as **Included**, `fulltext_available = True`
- Records with no open-access copy: set to **Full Text Needed** → you upload manually
- A **PRISMA snapshot** is saved automatically before the run begins (captures post-human-verification counts)

**Before running:** make sure the server is up (Cell 5). This runs in the background; poll `download_status()` to monitor.

**After running:** call `manual_list()` to see which papers need manual upload, then use the UI or Cell 4 to upload them.

### Cell 8d-fix — Clear bad PMC cache + re-download

The `.txt` files downloaded via PMC XML before this fix contained raw
metadata junk instead of clean article text. Run this cell **once** to
delete those files, then re-run Cell 8d to fetch them properly.

> **Skip if** you haven't run the full-text download yet — the fix is
> already applied and the downloader will save clean text from now on.

In [ ]:
# ── Delete bad PMC .txt cache files so they are re-fetched cleanly ──────
# Detection: JATS-XML-derived files always contain PMC\d{7,} in the first
# 400 chars, or HTML entities like &amp;, or author contribution keywords.
# Safe to re-run: only deletes files that match at least one indicator.

import os, re
from pathlib import Path

PDF_DIR = Path('/content/sysrev/systematic-review/data/pdfs')

deleted = []
kept    = []

for txt_file in sorted(PDF_DIR.glob('*.txt')):
    try:
        # Read first 600 chars only — metadata always appears at the very start
        sample = txt_file.read_text(encoding='utf-8', errors='replace')[:600]
    except Exception:
        continue

    is_bad = (
        # PMC article ID in header (most reliable: all JATS files have this)
        bool(re.search(r'PMC\d{5,}', sample))
        # Explicit PMC metadata fields
        or 'pmc-status-' in sample
        or 'pmc-prop-'   in sample
        or 'pmc-license-ref' in sample
        # Unescaped HTML entity — sign of XML stripped to text
        or '&amp;' in sample
        # JATS author contribution statement (always in the metadata block)
        or 'Writing – original draft' in sample
        or 'Writing - original draft' in sample
    )

    if is_bad:
        txt_file.unlink()
        deleted.append(txt_file.name)
    else:
        kept.append(txt_file.name)

print(f'Deleted {len(deleted)} bad .txt files (JATS/PMC metadata junk):')
for name in deleted[:30]:
    print(f'  ✗ {name}')
if len(deleted) > 30:
    print(f'  ... and {len(deleted) - 30} more')

print(f'\nKept {len(kept)} .txt files (content looks clean)')
for name in kept[:10]:
    print(f'  ✓ {name}')

print()
if deleted:
    print(f'Next steps:')
    print(f'  1. Re-run Cell 1 (git pull) to get the latest downloader fix')
    print(f'  2. Re-run Cell 19 to RESTART the server with the fixed code')
    print(f'  3. Re-run Cell 8d to re-download {len(deleted)} file(s)')
    print()
    print('IMPORTANT: Step 2 (server restart) is required so the running')
    print('           server loads the fixed _pmc_html_text() function.')
else:
    print('No bad files found — all .txt files look clean already.')


In [ ]:
import requests, time, json as _json

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

# ── Optional: your email improves Unpaywall/OpenAlex rate limits ──
CONTACT_EMAIL = 'your.email@institution.edu'  # change or leave blank


def save_prisma_snapshot(label='manual'):
    """Save a PRISMA flow snapshot at the current pipeline state."""
    r = requests.post(f'{BASE}/prisma/snapshot', json={'label': label})
    snap = r.json().get('snapshot', {})
    counts = snap.get('counts', {})
    print(f'PRISMA snapshot saved: {snap.get("label")}')
    print(f'  Identified: {counts.get("identified", "?")} | After dedup: {counts.get("after_dedup", "?")} | Title-included: {counts.get("title_abstract_included", "?")} | Full-text needed: {counts.get("fulltext_needed", "?")} | Final included: {counts.get("final_included", "?")} | Excluded: {counts.get("title_abstract_excluded", "?")} | Needs human: {counts.get("needs_human_verification", "?")}')
    return snap


def start_download(email=CONTACT_EMAIL, limit=None):
    """Trigger the auto-download background job."""
    payload = {'email': email or None, 'limit': limit}
    r = requests.post(f'{BASE}/fulltext/download', json=payload)
    print('Download started:', r.json())


def download_status():
    """Poll download progress."""
    r = requests.get(f'{BASE}/fulltext/download-status')
    s = r.json()
    if s.get('running'):
        print('Status: RUNNING…')
    else:
        log = s.get('last_run', {})
        print(f'Status: COMPLETE')
        print(f'  Auto-downloaded : {log.get("auto_downloaded", 0)}')
        print(f'  Manual needed   : {log.get("manual_needed", 0)}')
        print(f'  Completed at    : {log.get("completed_at", "—")}')
    return s


def manual_list(show=20):
    """Print the first `show` papers that need manual PDF upload."""
    s = requests.get(f'{BASE}/fulltext/download-status').json()
    records = s.get('manual_needed', [])
    print(f'{len(records)} papers need manual upload (showing first {show}):')
    for i, r in enumerate(records[:show], 1):
        print(f'  {i:3}. [{r.get("record_id","")[:8]}] {r.get("title","")[:70]}')
        print(f'       DOI: {r.get("doi","—")}  |  URL: {r.get("url","—")[:60]}')
    return records


def export_manual_csv():
    """Download CSV list of papers needing manual retrieval to Colab."""
    from google.colab import files as colab_files
    import urllib.request
    csv_url = f'http://localhost:{PORT}/api/fulltext/manual-list/csv'
    urllib.request.urlretrieve(csv_url, '/content/manual_fulltext_needed.csv')
    colab_files.download('/content/manual_fulltext_needed.csv')
    print('CSV downloaded — see manual_fulltext_needed.csv')


# ── STEP 0: exclude corrupt-cache papers + write recovery log ────────────
# These papers have .txt files with completely wrong content (wrong paper's
# text was cached). Excluding them keeps the final set clean.
# Their details are saved to data/corrupt_excluded.json for manual recovery.
import json as _json2
from pathlib import Path as _Path

APP_DIR = '/content/sysrev/systematic-review'
CORRUPT_LOG = _Path(f'{APP_DIR}/data/corrupt_excluded.json')
CORRUPT_LOG.parent.mkdir(parents=True, exist_ok=True)

CORRUPT_PAPERS = [
    {
        'record_id': '1d45cad8-4f7c-427c-8e8d-5032438dd4e2',
        'title':     'SFT-TA: Supervised Fine-Tuned Agents in Multi-Agent LLMs for Automated Inductive Coding',
        'reason':    'Corrupt cache: .txt is 4.4 MB of unrelated cardiology conference abstracts (ESOC25)',
        'doi':       '',
        'note':      'Find correct PDF via Semantic Scholar or Google Scholar and upload manually.',
    },
    {
        'record_id': 'bd433098-3f76-45c1-b0d4-30787ea88f60',
        'title':     'Reconciling Methodological Paradigms: Employing Large Language Models as Novice Researchers',
        'reason':    'Corrupt cache: .txt is 2.8 MB of unrelated nursing residency abstracts',
        'doi':       '',
        'note':      'Find correct PDF via Semantic Scholar or Google Scholar and upload manually.',
    },
]

# Write / merge log
existing = []
if CORRUPT_LOG.exists():
    try:
        existing = _json2.loads(CORRUPT_LOG.read_text())
    except Exception:
        existing = []
existing_ids = {e['record_id'] for e in existing}
added = [e for e in CORRUPT_PAPERS if e['record_id'] not in existing_ids]
with open(CORRUPT_LOG, 'w') as _f:
    _json2.dump(existing + added, _f, indent=2)

# Exclude via API
for entry in CORRUPT_PAPERS:
    bad_id   = entry['record_id']
    rationale = entry['reason']
    r = requests.post(
        f'{BASE}/records/{bad_id}/verify',
        json={'decision': 'Excluded', 'rationale': rationale, 'reviewer': 'human'}
    )
    short = bad_id[:8]
    if r.status_code == 200:
        print(f'✓ Excluded {short}… — {entry["title"][:55]}')
    elif r.status_code == 404:
        print(f'  {short}… not found (may already be excluded)')
    else:
        print(f'  Warning {short}…: {r.status_code} — {r.text[:80]}')

print(f'\nCorrupt-excluded log: {CORRUPT_LOG}  ({len(existing + added)} entr{"y" if len(existing+added)==1 else "ies"})')
print('Run Cell 8g to see recovery options.\n')

# ── STEP 1: save a PRISMA snapshot at post-human-verification stage ────────
print('=== Step 1: Saving PRISMA snapshot (post human verification) ===')
save_prisma_snapshot('post_human_verification')

# ── STEP 2: start auto-download ────────────────────────────────────────────
print('\n=== Step 2: Starting auto-download ===')
start_download(email=CONTACT_EMAIL)

# ── STEP 3: monitor (run in a loop or call download_status() manually) ─────
print('\n=== Monitoring download (checking every 30s, up to 30 min) ===')
for _ in range(60):        # 60 × 30 s = 30 min
    time.sleep(30)
    s = download_status()
    if not s.get('running'):
        break

# ── STEP 4: see what needs manual upload ───────────────────────────────────
print('\n=== Step 4: Papers needing manual PDF upload ===')
manual_list(show=30)

# ── STEP 5: export CSV for manual retrieval ────────────────────────────────
# Uncomment to download the CSV:
# export_manual_csv()


### Cell 8e — Full sanity-check of all included papers (before GPT)

Scans **every** included paper that has a file on disk.

For each paper it checks:
- **Size** — `.txt` files >500 k chars are suspicious (corrupt PMC downloads are typically 2–5 MB)
- **Title-keyword match** — are significant words from the paper title present in the first 5 k chars of content?
- **Junk patterns** — known JATS/PMC metadata markers, HTML entities, unrelated medical keywords

Prints a one-line status for every paper, then full detail for anything flagged **SUSPICIOUS** or **JUNK**.  
Saves flags to `data/suspect_papers.json` for use in Cell 8g (manual recovery).

> Run this **before** `run_stage('extraction')`.  Re-run any time after re-downloading files.

In [ ]:
# ── Full sanity-check: scan all included papers before GPT ──────────────
import sys, json, re
from pathlib import Path

APP_DIR = '/content/sysrev/systematic-review'
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

STATE_FILE   = f'{APP_DIR}/data/pipeline_state.jsonl'
PDF_DIR      = Path(f'{APP_DIR}/data/pdfs')
SUSPECT_FILE = Path(f'{APP_DIR}/data/suspect_papers.json')

PREVIEW_CHARS = 1500

# ── Junk detection ────────────────────────────────────────────────────
# PMC metadata dumps have the PMC ID in the VERY FIRST ~60 chars.
# Legitimate PMC full-text articles may contain a PMC ID further in,
# so only flag when it appears before char 60 to avoid false positives.
def _junk_flags(sample: str) -> list:
    flags = []
    head60 = sample[:60]   # real JATS dumps: PMC ID is the very first token
    rest   = sample[60:]
    if re.search(r'PMC\d{5,}', head60):
        flags.append('PMC ID at start — likely JATS metadata dump')
    if '&amp;' in sample:
        flags.append('HTML entity &amp;')
    if re.search(r'Writing [–-] original draft', sample):
        flags.append('JATS author contribution statement')
    if 'pmc-status-' in sample or 'pmc-prop-' in sample:
        flags.append('PMC property tag')
    return flags

MEDICAL_SPAM = [
    'CHA2DS2-VASc', 'atrial fibrillation', 'ischaemic stroke',
    'oncology nurse', 'abdominal percussion', 'antithrombotic',
    'haemorr', 'carotid filter', 'NIHSS',
]

def _read_sample(path: Path, n=5000) -> str:
    try:
        if path.suffix == '.txt':
            with path.open(encoding='utf-8', errors='replace') as f:
                return f.read(n)
        import fitz
        doc = fitz.open(str(path))
        text = ''
        for pg in doc:
            text += pg.get_text()
            if len(text) >= n:
                break
        return text[:n]
    except Exception:
        return ''

def _title_keywords(title: str):
    stop = {'a','an','the','of','in','on','at','to','for','and',
            'or','with','using','via','from','by','as','is','are',
            'this','that','we','our','its','into','based','large'}
    return [w for w in re.sub(r'[^a-z0-9 ]', ' ', title.lower()).split()
            if len(w) > 3 and w not in stop]

def _title_hit_ratio(keywords, sample: str) -> float:
    s = sample.lower()
    if not keywords:
        return 1.0
    hits = sum(1 for k in keywords if k in s)
    return hits / len(keywords)

def _file_size_bytes(path: Path) -> int:
    try:
        return path.stat().st_size
    except Exception:
        return 0

# ── Load pipeline state ────────────────────────────────────────────────
records = []
with open(STATE_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# ── Split into: has-file / no-file ────────────────────────────────────
has_file  = []   # (record_id, title, pdf_path, doi)
no_file   = []   # (record_id, title, doi) — included but nothing on disk

for rec in records:
    fd = rec.get('final_decision') or ''
    if fd != 'Included':
        continue
    screened  = rec.get('screened') or {}
    pdf_path  = screened.get('pdf_path') or rec.get('pdf_path')
    title = (
        screened.get('title')
        or (rec.get('dedup') or {}).get('title')
        or (rec.get('raw')   or {}).get('title')
        or rec.get('title') or '(unknown)')
    doi = ((rec.get('dedup') or {}).get('doi')
           or (rec.get('raw')  or {}).get('doi') or '')
    rid = rec.get('record_id', '?')

    if pdf_path and Path(pdf_path).exists():
        has_file.append((rid, title, pdf_path, doi))
    else:
        no_file.append((rid, title, doi, pdf_path or ''))

total_included = len(has_file) + len(no_file)
print(f'Included papers : {total_included}')
print(f'  With file     : {len(has_file)}')
print(f'  Missing file  : {len(no_file)}  ← need manual upload\n')

# ── Scan papers that have files ────────────────────────────────────────
OK = []; SUSPICIOUS = []; JUNK = []

for record_id, title, pdf_path, doi in has_file:
    p = Path(pdf_path)
    sample = _read_sample(p)

    junk_hits    = _junk_flags(sample)
    medical_hits = [w for w in MEDICAL_SPAM if w.lower() in sample.lower()]

    size_bytes = _file_size_bytes(p)
    big        = p.suffix == '.txt' and size_bytes > 500_000

    kws       = _title_keywords(title)
    hit_ratio = _title_hit_ratio(kws, sample)

    is_junk = bool(junk_hits) or len(medical_hits) >= 2
    is_susp = (big or hit_ratio < 0.15) and not is_junk

    entry = {
        'record_id':   record_id,
        'title':       title,
        'doi':         doi,
        'file':        str(p),
        'ext':         p.suffix,
        'size_kb':     round(size_bytes / 1024),
        'hit_ratio':   round(hit_ratio, 2),
        'junk_hits':   junk_hits,
        'medical_hits':medical_hits,
        'sample_100':  sample[:100].replace('\n', ' '),
    }
    if is_junk:
        JUNK.append(entry)
    elif is_susp:
        SUSPICIOUS.append(entry)
    else:
        OK.append(entry)

# ── Summary table ──────────────────────────────────────────────────────
W = 72
print(f"  {'#':>3}  {'ST':4}  {'EXT':4}  {'KB':>6}  {'KW%':>4}  Title")
print('  ' + '─' * W)
all_sorted = [('✅', e) for e in OK] + [('⚠️', e) for e in SUSPICIOUS] + [('❌', e) for e in JUNK]
for i, (sym, e) in enumerate(all_sorted, 1):
    kw_pct = f"{int(e['hit_ratio']*100):>3}%"
    print(f"  {i:>3}  {sym:4}  {e['ext']:4}  {e['size_kb']:>6}  {kw_pct}  {e['title'][:70]}")

print()
print(f'Summary: {len(OK)} OK  |  {len(SUSPICIOUS)} suspicious  |  {len(JUNK)} junk')

# ── Detail for flagged papers ──────────────────────────────────────────
if SUSPICIOUS or JUNK:
    SEP = '═' * 70
    print('\n' + SEP)
    print('FLAGGED PAPERS — review before running GPT extraction')
    print(SEP)
    for e in SUSPICIOUS + JUNK:
        status = '❌ JUNK' if e in JUNK else '⚠️  SUSPICIOUS'
        print(f'\n{status}')
        print(f"  ID      : {e['record_id']}")
        print(f"  Title   : {e['title'][:80]}")
        print(f"  DOI     : {e['doi'] or '—'}")
        print(f"  File    : {Path(e['file']).name}  ({e['ext']}  {e['size_kb']:,} KB)")
        print(f"  KW match: {int(e['hit_ratio']*100)}%  |  junk={e['junk_hits']}  |  medical={e['medical_hits'][:3]}")
        print(f"  Content preview (first 200 chars):")
        print(f"    {e['sample_100'][:200]}")
else:
    print('\n✓ No suspicious or junk files detected.')

# ── Papers with NO file on disk ────────────────────────────────────────
if no_file:
    SEP = '═' * 70
    print('\n' + SEP)
    print(f'  {len(no_file)} INCLUDED PAPERS WITH NO FILE ON DISK')
    print('  These need manual PDF upload before extraction can run.')
    print(SEP)
    for i, (rid, title, doi, hint) in enumerate(no_file, 1):
        title_q  = '+'.join(title.split()[:8])
        doi_link = f'https://doi.org/{doi}' if doi else '—'
        ss_link  = f'https://www.semanticscholar.org/search?q={title_q}&sort=Relevance'
        print(f'  {i:>2}. [{rid[:8]}] {title[:70]}')
        print(f'         DOI    : {doi_link}')
        print(f'         Search : {ss_link}')
        if hint:
            print(f'         Hint   : download expected at {Path(hint).name}')
        print()
else:
    print('\n✓ All included papers have a file on disk.')

# ── Save suspect list for Cell 8g ─────────────────────────────────────
suspect_all = SUSPICIOUS + JUNK
SUSPECT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(SUSPECT_FILE, 'w') as f:
    json.dump(suspect_all, f, indent=2)

if suspect_all:
    print(f'Suspect list saved → {SUSPECT_FILE}')
    print(f'Run Cell 8g for recovery options.')


## Cell 8f — PRISMA 2020 Visual Flowchart

Generates a publication-quality **PRISMA 2020 flow diagram** from live pipeline data.

Run **after** Cell 8d (full-text download) so all counts are final.
Re-run any time to refresh the chart.

In [ ]:
import requests, matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

# ── Fetch live counts ──────────────────────────────────────────────────────
r = requests.get(f'{BASE}/prisma/report')
data = r.json()
p    = data['prisma_counts']
dbs  = data.get('source_db_counts_all', {})
r2   = data.get('round2_exclusion_reasons', [])

# Key numbers
n_identified    = p.get('identified', 0)
n_dupes         = p.get('duplicates_removed', 0)
n_after_dedup   = p.get('after_dedup', 0)
n_ta_excl       = p.get('title_abstract_excluded', 0)
n_ta_incl       = p.get('title_abstract_included', 0)
n_ft_retr       = p.get('fulltext_retrieved', 0)
n_ft_notret     = p.get('full_text_needed', 0)
n_r1_excl       = p.get('fulltext_r1_excluded', 0)
n_r1_passed     = p.get('fulltext_r1_passed', 0)
n_r2_excl       = p.get('fulltext_r2_excluded', 0)
n_final         = p.get('final_included', 0)

# DB string (top 4 by count)
db_parts = sorted(dbs.items(), key=lambda x: -x[1])[:4]
db_str   = '\n'.join(f'{k}: n={v}' for k, v in db_parts)
if len(dbs) > 4:
    others = sum(v for k, v in sorted(dbs.items(), key=lambda x: -x[1])[4:])
    db_str += f'\nOther: n={others}'

# R2 exclusion reasons (top 5)
r2_lines = [f"  • {x['label']} (n={x['count']})" for x in r2 if x['count'] > 0][:5]
r2_str   = '\n'.join(r2_lines) if r2_lines else '  (none)'

# ── Layout ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 17))
ax.set_xlim(0, 13)
ax.set_ylim(0, 17)
ax.axis('off')

BLUE   = '#1A5276'
LBLUE  = '#D6EAF8'
GREEN  = '#1E8449'
LGREEN = '#D5F5E3'
ORANGE = '#E67E22'
LORANG = '#FDEBD0'
RED    = '#C0392B'
LRED   = '#FADBD8'
GRAY   = '#717D7E'
LGRAY  = '#F2F3F4'

def box(ax, x, y, w, h, text, fc=LBLUE, ec=BLUE, fontsize=9, bold=False):
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.08', facecolor=fc,
                          edgecolor=ec, linewidth=1.4, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold' if bold else 'normal',
            wrap=True, zorder=4,
            multialignment='center',
            bbox=dict(boxstyle='square,pad=0', fc='none', ec='none'))

def side_box(ax, x, y, w, h, text, fc=LRED, ec=RED, fontsize=8.5):
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.08', facecolor=fc,
                          edgecolor=ec, linewidth=1.2, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            wrap=True, zorder=4, multialignment='center',
            bbox=dict(boxstyle='square,pad=0', fc='none', ec='none'))

def arrow(ax, x1, y1, x2, y2, col=BLUE):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=col, lw=1.6), zorder=5)

def h_arrow(ax, x1, y1, x2, y2, col=RED):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=col, lw=1.4,
                                connectionstyle='arc3,rad=0'), zorder=5)

# ── Column centres ─────────────────────────────────────────────────────────
CX = 5.0     # main flow centre
SX = 10.5    # exclusion boxes centre
W  = 6.5     # main box width
SW = 4.8     # side box width

# ── Title ──────────────────────────────────────────────────────────────────
ax.text(CX, 16.6, 'PRISMA 2020 Flow Diagram',
        ha='center', va='center', fontsize=14, fontweight='bold', color=BLUE)
ax.text(CX, 16.25, 'LLMs in Qualitative Data Analysis — Systematic Review',
        ha='center', va='center', fontsize=9.5, color=GRAY, style='italic')

# ── Box 1: Identification ──────────────────────────────────────────────────
Y1 = 15.5
box(ax, CX, Y1, W, 0.9,
    f'Records identified from databases\n{db_str}\n\nTotal: n = {n_identified}',
    fc=LBLUE, ec=BLUE, fontsize=8.5)

# ── Arrow down ────────────────────────────────────────────────────────────
arrow(ax, CX, Y1-0.45, CX, 14.35)
# Side: duplicates removed
side_box(ax, SX, 14.7, SW, 0.55,
         f'Duplicates removed: n = {n_dupes}',
         fc=LORANG, ec=ORANGE)
h_arrow(ax, CX+W/2, 14.7, SX-SW/2, 14.7, col=ORANGE)

# ── Box 2: After dedup ─────────────────────────────────────────────────────
Y2 = 14.0
box(ax, CX, Y2, W, 0.65,
    f'Records after deduplication: n = {n_after_dedup}',
    fc=LBLUE, ec=BLUE)

arrow(ax, CX, Y2-0.325, CX, 13.05)

# Side: T/A excluded
side_box(ax, SX, 13.35, SW, 0.55,
         f'Excluded at title/abstract screening: n = {n_ta_excl}',
         fc=LRED, ec=RED)
h_arrow(ax, CX+W/2, 13.35, SX-SW/2, 13.35)

# ── Box 3: Full-text assessed ──────────────────────────────────────────────
Y3 = 12.7
box(ax, CX, Y3, W, 0.85,
    f'Full-text assessed for eligibility: n = {n_ta_incl}\n'
    f'  Retrieved: n = {n_ft_retr}   │   Not obtained: n = {n_ft_notret}',
    fc=LBLUE, ec=BLUE, fontsize=8.5)

arrow(ax, CX, Y3-0.425, CX, 11.65)

# Section label
ax.text(CX, 11.95, '─── Round-1 Full-text Screening ───',
        ha='center', va='center', fontsize=8.5, color=GRAY, style='italic')

# Side: R1 excluded
side_box(ax, SX, 11.4, SW, 0.55,
         f'Excluded at round-1: n = {n_r1_excl}',
         fc=LRED, ec=RED)
h_arrow(ax, CX+W/2, 11.4, SX-SW/2, 11.4)

# ── Box 4: Passed R1 ──────────────────────────────────────────────────────
Y4 = 11.1
box(ax, CX, Y4, W, 0.65,
    f'Passed round-1 (assessed for round-2): n = {n_r1_passed}',
    fc=LBLUE, ec=BLUE)

arrow(ax, CX, Y4-0.325, CX, 9.9)

ax.text(CX, 10.2, '─── Round-2 Full-text Screening (refined criteria) ───',
        ha='center', va='center', fontsize=8.5, color=GRAY, style='italic')

# Side: R2 excluded
r2_n_lines = 1 + len([x for x in r2 if x['count'] > 0][:5])
r2_box_h   = max(0.7, r2_n_lines * 0.28 + 0.3)
side_box(ax, SX, 10.0 - r2_box_h/2 + 0.3, SW, r2_box_h,
         f'Excluded at round-2: n = {n_r2_excl}\n{r2_str}',
         fc=LRED, ec=RED, fontsize=8)
h_arrow(ax, CX+W/2, 9.7, SX-SW/2, 9.7)

# ── Box 5: Final included ─────────────────────────────────────────────────
Y5 = 9.2
box(ax, CX, Y5, W, 0.85,
    f'Studies included in final synthesis\n\nn = {n_final}',
    fc=LGREEN, ec=GREEN, bold=True, fontsize=11)

arrow(ax, CX, Y3-0.425 - (Y3-0.425 - (Y4+0.325)), CX, Y4+0.325)
arrow(ax, CX, Y4-0.325, CX, Y5+0.425)

# ── Footer ─────────────────────────────────────────────────────────────────
ax.text(CX, 8.6, 'Generated automatically from pipeline_state.jsonl via /api/prisma/report',
        ha='center', fontsize=7.5, color=GRAY, style='italic')

plt.tight_layout()
plt.savefig('/content/prisma_flowchart.png', dpi=180, bbox_inches='tight',
            facecolor='white')
plt.show()
print(f'\n✓ Saved: /content/prisma_flowchart.png')
print(f'  Final included: {n_final}')

# Also print the ASCII version for quick reference
print()
r2 = requests.get(f'{BASE}/prisma/flowchart')
print(r2.text)


### Cell 8g — Manual Recovery: corrupt-excluded papers

Papers that were excluded because their cached file contained completely wrong content
(another paper's text was accidentally saved with the correct filename).

This cell:
1. Lists all papers in `data/corrupt_excluded.json`  
2. Shows DOI + search links so you can find the correct PDF manually  
3. Gives a one-liner to **re-include** a paper after uploading the correct PDF

> After uploading the correct PDF via the server UI, call `re_include(record_id)` at the bottom.

In [ ]:
# ── Manual recovery for corrupt-excluded papers ──────────────────────
import json, requests
from pathlib import Path

APP_DIR      = '/content/sysrev/systematic-review'
CORRUPT_LOG  = Path(f'{APP_DIR}/data/corrupt_excluded.json')
SUSPECT_FILE = Path(f'{APP_DIR}/data/suspect_papers.json')

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

def _load(path):
    if not path.exists():
        return []
    try:
        return json.loads(path.read_text())
    except Exception:
        return []

corrupt  = _load(CORRUPT_LOG)
suspects = _load(SUSPECT_FILE)

# Merge into one recovery list (avoid duplicates by record_id)
seen = set()
recovery = []
for e in corrupt:
    if e['record_id'] not in seen:
        seen.add(e['record_id'])
        recovery.append({**e, 'source': 'corrupt_excluded'})
for e in suspects:
    if e['record_id'] not in seen:
        seen.add(e['record_id'])
        recovery.append({
            'record_id': e['record_id'],
            'title':     e['title'],
            'doi':       e.get('doi', ''),
            'reason':    f"Flagged by sanity-check: size={e['size_kb']} KB, kw_match={int(e['hit_ratio']*100)}%",
            'note':      'Check file content; if wrong, find correct PDF and re-upload.',
            'source':    'suspect_scan',
        })

if not recovery:
    print('No corrupt/suspect papers on record — nothing to recover.')
else:
    SEP = '═' * 70
    print(f'{SEP}')
    print(f'  {len(recovery)} paper(s) need manual recovery')
    print(f'{SEP}\n')

    for i, e in enumerate(recovery, 1):
        rid   = e['record_id']
        title = e['title']
        doi   = e.get('doi', '') or ''
        reason = e.get('reason', '')
        note   = e.get('note', '')
        src    = e.get('source', '')

        # Build search links
        title_q = '+'.join(title.split()[:8])
        ss_link  = f'https://www.semanticscholar.org/search?q={title_q}&sort=Relevance'
        gs_link  = f'https://scholar.google.com/scholar?q={title_q}'
        doi_link = f'https://doi.org/{doi}' if doi else '—'

        print(f'  [{i}] {title[:75]}')
        print(f'        ID     : {rid}')
        print(f'        Reason : {reason[:80]}')
        print(f'        DOI    : {doi_link}')
        print(f'        Search : {ss_link}')
        print(f'        Note   : {note}')
        print()

# ── Re-include helper ─────────────────────────────────────────────────
def re_include(record_id: str, rationale: str = 'Correct PDF uploaded manually; re-included.'):
    """Mark a previously-excluded paper as Included again.
    Call this after uploading the correct PDF via the server UI.
    """
    r = requests.post(
        f'{BASE}/records/{record_id}/verify',
        json={'decision': 'Included', 'rationale': rationale, 'reviewer': 'human'}
    )
    if r.status_code == 200:
        print(f'✓ Re-included {record_id[:8]}…')
    else:
        print(f'Error {r.status_code}: {r.text[:100]}')

print('─' * 70)
print('To re-include a paper after uploading the correct PDF:')
print('  re_include("record_id_here")')
print()
print('To flag additional papers found during Cell 8e scan:')
print('  See data/suspect_papers.json — edit and re-run this cell.')


### Cell 8h — Upload replacement PDFs for junk / missing papers

For each paper listed below:
1. Obtain the correct PDF (via DOI, Semantic Scholar, or your library)
2. Rename the file to exactly `<record-id>.pdf` (the UUID shown)
3. Set `UPLOAD = True` and re-run — a file picker will open
4. Select **all** renamed PDFs at once and upload
5. Re-run **Cell 8e** to confirm everything is clean

In [ ]:
# ── Cell 8h: Upload replacement PDFs for junk / missing papers ───────────
import json, re, shutil
from pathlib import Path

UPLOAD = False   # ← set True when your PDFs are renamed and ready

APP_DIR       = '/content/sysrev/systematic-review'
STATE_FILE    = Path(f'{APP_DIR}/data/pipeline_state.jsonl')
PDF_DIR       = Path(f'{APP_DIR}/data/pdfs')
SUSPECT_FILE  = Path(f'{APP_DIR}/data/suspect_papers.json')
DRIVE_PDF_DIR = Path('/content/drive/MyDrive/SysRev_LLMs_QDA/pdfs')
DRIVE_STATE   = Path('/content/drive/MyDrive/SysRev_LLMs_QDA/pipeline_state.jsonl')

# ── Load state ────────────────────────────────────────────────────────
records = []
with open(STATE_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# ── Build list of papers needing upload ───────────────────────────────
need_upload = []

# 1. Junk files detected by Cell 8e
if SUSPECT_FILE.exists():
    suspects = json.loads(SUSPECT_FILE.read_text())
    for e in suspects:
        need_upload.append({
            'record_id':    e['record_id'],
            'title':        e['title'],
            'doi':          e.get('doi', ''),
            'old_file':     e['file'],           # corrupt file to delete
            'save_as':      str(PDF_DIR / f"{e['record_id']}.pdf"),
            'reason':       'junk — wrong article content downloaded',
        })

# 2. Included papers with no file on disk
junk_ids = {e['record_id'] for e in need_upload}
for rec in records:
    if rec.get('final_decision') != 'Included':
        continue
    if rec.get('record_id') in junk_ids:
        continue
    screened = rec.get('screened') or {}
    pdf_path = screened.get('pdf_path') or rec.get('pdf_path')
    if pdf_path and Path(pdf_path).exists():
        continue
    title = (screened.get('title')
             or (rec.get('dedup') or {}).get('title')
             or (rec.get('raw')   or {}).get('title')
             or rec.get('title') or '(unknown)')
    doi = ((rec.get('dedup') or {}).get('doi')
           or (rec.get('raw')  or {}).get('doi') or '')
    need_upload.append({
        'record_id': rec.get('record_id'),
        'title':     title,
        'doi':       doi,
        'old_file':  None,
        'save_as':   str(PDF_DIR / f"{rec.get('record_id')}.pdf"),
        'reason':    'no file on disk',
    })

# ── Print the checklist ───────────────────────────────────────────────
SEP = '═' * 70
print(f"{SEP}")
print(f"  {len(need_upload)} papers need PDFs")
print(f"  Rename each PDF to exactly  <ID>.pdf  before uploading")
print(f"{SEP}\n")

for i, e in enumerate(need_upload, 1):
    rid = e['record_id']
    doi_link = f"https://doi.org/{e['doi']}" if e['doi'] else '—'
    title_q  = '+'.join(e['title'].split()[:8])
    ss_link  = f"https://www.semanticscholar.org/search?q={title_q}&sort=Relevance"
    print(f"  {i:>2}. Rename → {rid}.pdf")
    print(f"       Title  : {e['title'][:72]}")
    print(f"       DOI    : {doi_link}")
    print(f"       Search : {ss_link}")
    print(f"       Reason : {e['reason']}")
    print()

if not UPLOAD:
    print("Set  UPLOAD = True  then re-run to open the upload dialog.")
else:
    from google.colab import files as colab_files
    print("Opening upload dialog — select all renamed PDFs at once…\n")
    uploaded = colab_files.upload()   # {filename: bytes}

    if not uploaded:
        print("No files uploaded.")
    else:
        lookup = {e['record_id']: e for e in need_upload}

        saved = []; skipped = []
        for fname, data in uploaded.items():
            # Strip Colab's ' (2)', ' (3)' etc. dedup suffix
            stem  = re.sub(r'\s*\(\d+\)$', '', Path(fname).stem)
            entry = lookup.get(stem)
            if not entry:
                entry = next((e for e in need_upload
                              if e['record_id'].startswith(stem)), None)
            if not entry:
                skipped.append(fname)
                continue

            # ── Save to /content/ ─────────────────────────────────────
            dest = Path(entry['save_as'])   # always .pdf
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(data)

            # Delete old corrupt file from /content/ if different
            old = entry.get('old_file')
            if old:
                old_p = Path(old)
                if old_p.exists() and old_p != dest:
                    old_p.unlink()

            # ── Mirror to Google Drive (so it survives session reset) ─
            if DRIVE_PDF_DIR.exists():
                drive_dest = DRIVE_PDF_DIR / dest.name
                drive_dest.write_bytes(data)
                # Delete old corrupt file from Drive too
                if old:
                    drive_old = DRIVE_PDF_DIR / Path(old).name
                    if drive_old.exists() and drive_old != drive_dest:
                        drive_old.unlink()
            else:
                print(f"  ⚠ Drive folder not found ({DRIVE_PDF_DIR}) — "
                      f"PDF saved to /content/ only. Run Cell 7 to back up.")

            saved.append((entry['record_id'], entry['title'][:55], str(dest)))

        # ── Update pipeline_state.jsonl → pdf_path → new .pdf ────────
        if saved:
            saved_ids = {r[0] for r in saved}
            updated = []
            for rec in records:
                if rec.get('record_id') in saved_ids:
                    new_path = str(PDF_DIR / f"{rec['record_id']}.pdf")
                    screened = rec.get('screened') or {}
                    screened['pdf_path'] = new_path
                    rec['screened'] = screened
                    rec.pop('pdf_path', None)
                updated.append(rec)
            # Write /content/ state
            with open(STATE_FILE, 'w') as f:
                for r in updated:
                    f.write(json.dumps(r) + '\n')
            # ── Write Drive state immediately ─────────────────────────
            if DRIVE_STATE.parent.exists():
                shutil.copy2(STATE_FILE, DRIVE_STATE)
                print(f"✓ pipeline_state.jsonl saved to Drive.")
            else:
                print(f"  ⚠ Drive state path not found — run Cell 7 to back up state.")

        print(f"\n✓ Saved {len(saved)} PDF(s):")
        for rid, title, dest in saved:
            print(f"   [{rid[:8]}] {title}")
            print(f"              → {Path(dest).name} (+ Drive mirror)")
        if skipped:
            print(f"\n⚠ {len(skipped)} file(s) not matched — check naming:")
            for fn in skipped:
                print(f"   {fn}  (need UUID as filename)")

        print("\n→ Re-run Cell 8e to confirm all papers are clean.")


### Cell 8i — Export included-papers list & pilot extraction

**Part A** exports the 84 included papers as a CSV so you can verify titles / DOIs before any GPT call.

**Part B** runs extraction on **one paper only** (the one you pick by index) and prints the full result for you to review. When happy, use Cell 8j to extract all.

In [ ]:
# ── Cell 8i: Export included list + pilot extraction ─────────────────────
import json, csv, io, time, requests
from pathlib import Path
from google.colab import files as colab_files

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

try:
    requests.get(f'{BASE}/pipeline/status', timeout=3)
except Exception:
    print("❌ Server is not running — run Cell 5 first.")
    raise SystemExit

# ═══════════════════════════════════════════════════════════
# PART A — Export included papers list as CSV
# ═══════════════════════════════════════════════════════════
APP_DIR    = '/content/sysrev/systematic-review'
STATE_FILE = Path(f'{APP_DIR}/data/pipeline_state.jsonl')

records = []
with open(STATE_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

included = []
for rec in records:
    if rec.get('final_decision') != 'Included':
        continue
    screened = rec.get('screened') or {}
    dedup    = rec.get('dedup')    or {}
    raw      = rec.get('raw')      or {}
    title    = screened.get('title') or dedup.get('title') or raw.get('title') or '(unknown)'
    doi      = dedup.get('doi')  or raw.get('doi')  or ''
    year     = dedup.get('year') or raw.get('year') or ''
    authors  = dedup.get('authors') or raw.get('authors') or []
    author_str = '; '.join(authors[:3]) + (' et al.' if len(authors) > 3 else '')
    pdf_path = screened.get('pdf_path') or rec.get('pdf_path') or ''
    included.append({
        '#':         len(included) + 1,
        'record_id': rec.get('record_id', ''),
        'title':     title,
        'doi':       doi,
        'year':      year,
        'authors':   author_str,
        'file':      Path(pdf_path).name if pdf_path else '',
        'file_kb':   round(Path(pdf_path).stat().st_size/1024) if pdf_path and Path(pdf_path).exists() else '',
    })

print(f"{'═'*80}")
print(f"  {len(included)} included papers")
print(f"{'═'*80}")
print(f"  {'#':>3}  {'Year':>4}  {'Title':<65}  DOI")
print(f"  {'─'*3}  {'─'*4}  {'─'*65}  {'─'*20}")
for e in included:
    print(f"  {e['#']:>3}  {str(e['year']):>4}  {e['title'][:65]:<65}  {e['doi'][:40]}")

out = io.StringIO()
w = csv.DictWriter(out, fieldnames=['#','record_id','title','doi','year','authors','file','file_kb'])
w.writeheader(); w.writerows(included)
csv_path = f'{APP_DIR}/data/included_papers.csv'
with open(csv_path, 'w') as f:
    f.write(out.getvalue())
print(f"\nSaved → {csv_path}")
colab_files.download(csv_path)
print("✓ CSV downloaded.")

# ═══════════════════════════════════════════════════════════
# PART B — Pilot extraction on ONE paper
# Prints FOUR sections matching the four saved CSV files:
#   1. Agent 1 (GPT-5) raw extraction
#   2. Agent 2 (GPT-5.4-mini) raw extraction
#   3. Merged/final result
#   4. Disagreements table (field-level diff)
# ═══════════════════════════════════════════════════════════
RUN_PILOT  = False  # ← set True to run
TEST_INDEX = 1      # ← paper number (1-based)

HARD_FIELDS = {
    'model_name','workflow_structure','analytic_task','human_comparison',
    'fine_tuned','rag_used','formal_methodology','qualitative_approach',
}
SKIP = {'not_reported_fields','uncertain_fields','extraction_notes'}

def _fmt(v):
    if v is None:           return '—'
    if isinstance(v, list): return ', '.join(str(x) for x in v) if v else '—'
    s = str(v).strip()
    return s if s else '—'

def _section(title):
    print(f"\n{'═'*70}")
    print(f"  {title}")
    print(f"{'═'*70}")

def _print_extraction(label, d):
    """Print all fields from one agent dict — mirrors extraction_agentN.csv."""
    uncertain = set(d.get('uncertain_fields') or [])
    nrf       = d.get('not_reported_fields') or []
    print(f"  {'Field':<33} {'Value':<70}  Notes")
    print(f"  {'─'*33} {'─'*70}  {'─'*10}")
    for k, v in d.items():
        if k in SKIP: continue
        lbl = k.replace('_', ' ').title()
        fv  = _fmt(v)
        notes = []
        if k in HARD_FIELDS:  notes.append('HARD')
        if k in uncertain:    notes.append('uncertain ⚠')
        if k in nrf:          notes.append('not reported')
        print(f"  {lbl:<33} {fv[:70]:<70}  {', '.join(notes)}")

def _print_merged(final, a1, a2, disagreements):
    """Print merged result — mirrors extraction_merged.csv."""
    disagree_set = set(disagreements or [])
    uncertain    = set((final or {}).get('uncertain_fields') or [])
    nrf          = (final or {}).get('not_reported_fields') or []
    print(f"  {'Field':<33} {'Merged value':<50}  {'Status'}")
    print(f"  {'─'*33} {'─'*50}  {'─'*20}")
    for k, v in (final or {}).items():
        if k in SKIP: continue
        lbl = k.replace('_', ' ').title()
        if k in disagree_set:
            hard = ' [HARD FIELD]' if k in HARD_FIELDS else ''
            print(f"  {lbl:<33} {'(null — disagreed)':<50}  ✗ agents differed{hard}")
        else:
            fv = _fmt(v)
            if fv == '—': continue  # not reported / null
            notes = []
            if k in HARD_FIELDS: notes.append('HARD')
            if k in uncertain:   notes.append('uncertain ⚠')
            print(f"  {lbl:<33} {fv[:50]:<50}  {', '.join(notes) or 'agreed ✓'}")

def _print_disagreements(a1, a2, final, disagreements):
    """Print field-level diff — mirrors extraction_disagreements.csv."""
    if not disagreements:
        print("  (none — agents agreed on all fields)")
        return
    print(f"  {'Field':<33} {'Agent1 (GPT-5)':<35} {'Agent2 (mini)':<35} {'Hard?'}")
    print(f"  {'─'*33} {'─'*35} {'─'*35} {'─'*5}")
    for field in disagreements:
        lbl  = field.replace('_', ' ').title()
        v1   = _fmt((a1 or {}).get(field))
        v2   = _fmt((a2 or {}).get(field))
        hard = '★ YES' if field in HARD_FIELDS else 'no'
        print(f"  {lbl:<33} {v1[:35]:<35} {v2[:35]:<35} {hard}")

# ── run ────────────────────────────────────────────────────────────────
if not RUN_PILOT:
    print("\n─────────────────────────────────────────────────────────────────────")
    print("Set  RUN_PILOT = True  and choose TEST_INDEX, then re-run.")
else:
    if TEST_INDEX < 1 or TEST_INDEX > len(included):
        print(f"TEST_INDEX out of range. Choose 1–{len(included)}.")
    else:
        chosen = included[TEST_INDEX - 1]
        print(f"\n{'═'*70}")
        print(f"Pilot extraction: #{TEST_INDEX} — {chosen['title'][:70]}")
        print(f"{'═'*70}")

        r = requests.post(
            f'{BASE}/pipeline/run',
            json={'stage': 'extraction', 'record_id': chosen['record_id']},
            timeout=300
        )
        if r.status_code != 200:
            print(f"Error: {r.status_code} {r.text[:200]}")
        else:
            resp = r.json()
            print(f"Status: {resp.get('status')}")
            if resp.get('status') == 'started':
                print("Polling…")
                for _ in range(60):
                    time.sleep(5)
                    if not requests.get(f'{BASE}/pipeline/status').json().get('running_stage'):
                        print("✓ Done.")
                        break
                    print(f"  {_*5}s…")

            rec_r = requests.get(f'{BASE}/records/{chosen["record_id"]}')
            if rec_r.status_code != 200:
                print(f"Could not fetch record: {rec_r.status_code}")
            else:
                rec_data      = rec_r.json()
                extracted     = rec_data.get('extracted') or {}
                a1            = extracted.get('extraction_agent1') or {}
                a2            = extracted.get('extraction_agent2') or {}
                final         = extracted.get('extraction_final') or {}
                disagreements = extracted.get('disagreement_fields') or []
                agrees        = extracted.get('agents_agree_extraction')

                if not final and not a1 and not a2:
                    print("No extraction result yet.")
                else:
                    # ── FILE 1: Agent 1 ──────────────────────────────────
                    _section("FILE 1 — AGENT 1 (GPT-5, primary extractor)")
                    if a1:
                        _print_extraction("Agent 1", a1)
                    else:
                        print("  (not available)")

                    # ── FILE 2: Agent 2 ──────────────────────────────────
                    _section("FILE 2 — AGENT 2 (GPT-5.4-mini, verification)")
                    if a2:
                        _print_extraction("Agent 2", a2)
                    else:
                        print("  (not available)")

                    # ── FILE 3: Merged ───────────────────────────────────
                    _section(f"FILE 3 — MERGED RESULT  (agents_agree={agrees})")
                    _print_merged(final, a1, a2, disagreements)
                    unc = (final or {}).get('uncertain_fields') or []
                    nrf = (final or {}).get('not_reported_fields') or []
                    if unc: print(f"\n  Uncertain fields  : {unc}")
                    if nrf: print(f"  Not reported      : {nrf}")

                    # ── FILE 4: Disagreements ────────────────────────────
                    _section(f"FILE 4 — DISAGREEMENTS  ({len(disagreements)} field(s))")
                    _print_disagreements(a1, a2, final, disagreements)
                    if disagreements:
                        hard_dis = [f for f in disagreements if f in HARD_FIELDS]
                        if hard_dis:
                            print(f"\n  ★ Hard-field disagreements → flagged for human review:")
                            for f in hard_dis:
                                print(f"    - {f}")

                    # ── QA score ─────────────────────────────────────────
                    _section("QA ASSESSMENT")
                    for k, v in (extracted.get('qa_score') or {}).items():
                        print(f"  {k:<40} {v}")

                    # ── Token usage + cost ────────────────────────────────
                    PRICE = {
                        "gpt-5":       {"in": 1.25,  "out": 10.00},
                        "gpt-5.4-mini":{"in": 0.75,  "out": 4.50},
                        "gpt-4o":      {"in": 2.50,  "out": 10.00},
                        "gpt-4o-mini": {"in": 0.15,  "out": 0.60},
                    }
                    token_usage = extracted.get('token_usage') or {}
                    if token_usage:
                        _section("TOKEN USAGE")
                        print(f"  {'Model':<20} {'Prompt':>10} {'Completion':>12} {'Cost (USD)':>12}")
                        print(f"  {'─'*20} {'─'*10} {'─'*12} {'─'*12}")
                        total_cost = 0.0
                        for model, cnts in token_usage.items():
                            p  = cnts.get('prompt_tokens', 0)
                            c  = cnts.get('completion_tokens', 0)
                            pr = PRICE.get(model, {"in": 0, "out": 0})
                            cost = (p * pr['in'] + c * pr['out']) / 1_000_000
                            total_cost += cost
                            print(f"  {model:<20} {p:>10,} {c:>12,} ${cost:>11.4f}")
                        print(f"  {'─'*56}")
                        print(f"  {'TOTAL':<44} ${total_cost:>11.4f}")
                        n_papers = 84
                        proj = total_cost * n_papers
                        print(f"\n  Projected for {n_papers} papers: ${proj:.2f}  "
                              f"(range ${proj*0.8:.2f}–${proj*1.3:.2f})")

                    print(f"\n{'═'*70}")
                    print("Review above. If correct → run Cell 8j to extract all 84 papers.")


### Cell 8j — Full extraction (all included papers)

Only run this **after** verifying the pilot in Cell 8i.
Extracts all 84 included papers using two independent GPT agents, then auto-merges results.
Papers where agents disagree on hard fields are flagged for human review.

In [ ]:
# ── Cell 8j: Full extraction — all included papers ────────────────────────
import requests, time, json, csv, sys
from pathlib import Path

PORT = 8000
BASE = f'http://localhost:{PORT}/api'
APP_DIR = '/content/sysrev/systematic-review'

sys.path.insert(0, APP_DIR)

def _server_ok():
    try:
        requests.get(f'{BASE}/pipeline/status', timeout=3)
        return True
    except Exception:
        print('Server not running — start Cell 5 first.')
        return False

if not _server_ok():
    raise SystemExit

# ── Pre-flight check ────────────────────────────────────────────────────
print("Pre-flight check:")
st = requests.get(f'{BASE}/pipeline/status').json()
c  = st.get('prisma_counts', {})
print(f"  Final included     : {c.get('final_included', 0)}")
print(f"  Already extracted  : {c.get('final_extracted', 0)}")
print(f"  Pending review     : {c.get('needs_human_extraction', 0)}")
print(f"  Running stage      : {st.get('running_stage') or 'none'}")
print()
print("Checkpoints: state + 4 CSV files saved every 5 papers → output/ + Drive.")
print("If interrupted, re-run — already-extracted papers are skipped.")


# ── Helper: write 4 checkpoint CSVs + download to computer ─────────────
def _save_and_download_csvs():
    """Read state file, build 4 CSVs, save to output/, download, sync to Drive."""
    from pipeline.models import PipelineRecord, PipelineStage

    STATE_FILE  = Path(f'{APP_DIR}/data/pipeline_state.jsonl')
    OUTPUT_DIR  = Path(f'{APP_DIR}/output')
    OUTPUT_DIR.mkdir(exist_ok=True)
    DRIVE_DIR   = Path('/content/drive/MyDrive/SysRev_LLMs_QDA')
    HARD_FIELDS = {
        'model_name', 'workflow_structure', 'analytic_task', 'human_comparison',
        'fine_tuned', 'rag_used', 'formal_methodology', 'qualitative_approach',
    }

    def _flat(v):
        if isinstance(v, list): return ' || '.join(str(x) for x in v)
        return v if v is not None else ''

    extracted_records = []
    with open(STATE_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            pr = PipelineRecord.model_validate_json(line)
            if pr.pipeline_stage == PipelineStage.EXTRACTION and pr.extracted is not None:
                extracted_records.append(pr)

    rows_a1, rows_a2, rows_merged, rows_diff = [], [], [], []
    for pr in extracted_records:
        ext   = pr.extracted
        title = (pr.dedup.title if pr.dedup else None) or (pr.raw.title if pr.raw else '') or ''
        base  = {
            'record_id': pr.record_id,
            'study_id':  pr.study_id or ext.study_id or '',
            'title':     title,
        }
        if ext.extraction_agent1:
            row = {**base}
            for k, v in ext.extraction_agent1.model_dump().items():
                row[k] = _flat(v)
            rows_a1.append(row)
        if ext.extraction_agent2:
            row = {**base}
            for k, v in ext.extraction_agent2.model_dump().items():
                row[k] = _flat(v)
            rows_a2.append(row)
        if ext.extraction_final:
            row = {**base}
            for k, v in ext.extraction_final.model_dump().items():
                row[k] = _flat(v)
            row['disagreement_fields'] = ' || '.join(ext.disagreement_fields)
            row['agents_agree']        = ext.agents_agree_extraction
            row['human_verified']      = ext.human_verified
            rows_merged.append(row)
        for field in ext.disagreement_fields:
            a1v = getattr(ext.extraction_agent1, field, None) if ext.extraction_agent1 else None
            a2v = getattr(ext.extraction_agent2, field, None) if ext.extraction_agent2 else None
            mgv = getattr(ext.extraction_final,  field, None) if ext.extraction_final  else None
            rows_diff.append({
                'record_id':     pr.record_id,
                'title':         title[:80],
                'field':         field,
                'agent1_gpt5':   _flat(a1v),
                'agent2_mini':   _flat(a2v),
                'merged':        _flat(mgv),
                'is_hard_field': field in HARD_FIELDS,
            })

    def _write(path, rows):
        if not rows: return 0
        with open(path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=rows[0].keys(), extrasaction='ignore')
            w.writeheader(); w.writerows(rows)
        return len(rows)

    files = {
        'extraction_agent1.csv':        (OUTPUT_DIR / 'extraction_agent1.csv',        rows_a1),
        'extraction_agent2.csv':        (OUTPUT_DIR / 'extraction_agent2.csv',        rows_a2),
        'extraction_merged.csv':        (OUTPUT_DIR / 'extraction_merged.csv',        rows_merged),
        'extraction_disagreements.csv': (OUTPUT_DIR / 'extraction_disagreements.csv', rows_diff),
    }

    print(f"\n{'─'*60}")
    print(f"  Saving {len(extracted_records)} extracted records to 4 CSV files")
    print(f"  Location: {OUTPUT_DIR}/")
    print(f"{'─'*60}")
    from google.colab import files as colab_files
    for fname, (path, rows) in files.items():
        n = _write(path, rows)
        if n:
            print(f"  ✓ {fname:<40} ({n} rows)  → downloading…")
            colab_files.download(str(path))
        else:
            print(f"  – {fname:<40} (no data)")

    # ── Sync to Drive ──────────────────────────────────────────────────
    import shutil
    if DRIVE_DIR.exists():
        synced = []
        for fname, (path, _) in files.items():
            if path.exists():
                shutil.copy2(str(path), str(DRIVE_DIR / fname))
                synced.append(fname)
        if synced:
            print(f"\n  ✓ Synced to Drive: {DRIVE_DIR}/")
            for f in synced:
                print(f"      {f}")
    else:
        print("\n  (Drive not mounted — files saved locally only)")

    print(f"\n  Summary: agent1={len(rows_a1)}, agent2={len(rows_a2)}, "
          f"merged={len(rows_merged)}, disagreements={len(rows_diff)}")


# ── Start extraction ────────────────────────────────────────────────────
CONFIRMED = True

if not CONFIRMED:
    print("\nSet  CONFIRMED = True  then re-run to begin extraction.")
else:
    r = requests.post(
        f'{BASE}/pipeline/run',
        json={'stage': 'extraction'},
        timeout=30
    )
    r.raise_for_status()
    resp = r.json()
    print(f"\nStarted: {resp.get('status')}")
    print("Extraction running in background. Polling every 30s…\n")
    print(f"  (Server auto-saves state + 4 CSV files to output/ every 5 papers)")
    print()

    start = time.time()
    try:
        while True:
            time.sleep(30)
            st      = requests.get(f'{BASE}/pipeline/status').json()
            running = st.get('running_stage', '')
            c       = st.get('prisma_counts', {})
            elapsed = int(time.time() - start)
            extracted = c.get('final_extracted', '?')
            pending   = c.get('needs_human_extraction', '?')
            total     = c.get('final_included', '?')
            print(f"  [{elapsed:>4}s]  stage={running or 'done ✓'}  "
                  f"extracted={extracted}/{total}  pending_review={pending}")

            if not running:
                log = st.get('stage_log', {}).get('extraction', {})
                print(f"\n✓ Extraction complete!  (elapsed {elapsed}s)")
                print(f"  Extracted  (agents agreed) : {log.get('extracted', extracted)}")
                print(f"  Pending human review       : {log.get('uncertain', pending)}")
                print(f"  Errors                     : {log.get('errors', '?')}")

                # ── Save & download all 4 CSV files ──────────────────────
                _save_and_download_csvs()

                print("\nNext steps:")
                print("  1. Cell 8k — download the evidence table CSV")
                print("  2. Cell 8l — push extraction CSVs to GitHub")
                print("  3. Web UI  — review pending-human-review records")
                break

    except KeyboardInterrupt:
        print("\nPolling stopped. Re-run to continue polling.")
        print("Already-extracted papers will NOT be re-processed.")
        print("You can also call  _save_and_download_csvs()  to save progress so far.")


### Cell 8k — Download evidence table CSV

Downloads the full evidence table (all extracted fields for all included papers) as a CSV file.

In [ ]:
# ── Cell 8k: Download evidence table ──────────────────────────────────────
import requests
from google.colab import files as colab_files
from pathlib import Path

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

r = requests.get(f'{BASE}/export/evidence-table/csv', timeout=30)
if r.status_code == 404:
    print("No extracted records yet — run Cell 8j first.")
elif r.status_code != 200:
    print(f"Error {r.status_code}: {r.text[:200]}")
else:
    out_path = '/content/sysrev/systematic-review/data/evidence_table.csv'
    with open(out_path, 'wb') as f:
        f.write(r.content)
    row_count = r.content.count(b'\n') - 1
    print(f"✓ Evidence table: {row_count} rows")
    colab_files.download(out_path)
    print("Downloaded.")

# Also print a quick summary table
r2 = requests.get(f'{BASE}/export/evidence-table', timeout=30)
if r2.status_code == 200:
    data = r2.json()
    rows = data.get('rows', [])
    if rows:
        print(f"\n{'─'*90}")
        print(f"  {'#':>3}  {'Study ID':>12}  {'Year':>4}  {'Model':<20}  {'Task':<25}  {'Title'[:30]}")
        print(f"  {'─'*3}  {'─'*12}  {'─'*4}  {'─'*20}  {'─'*25}  {'─'*30}")
        for i, row in enumerate(rows, 1):
            title = str(row.get('title', ''))[:30]
            model = str(row.get('model_name', '') or '—')[:20]
            task  = str(row.get('analytic_task', '') or '—')[:25]
            year  = str(row.get('year', '') or '—')
            sid   = str(row.get('study_id', '') or '—')
            print(f"  {i:>3}  {sid:>12}  {year:>4}  {model:<20}  {task:<25}  {title}")


### Cell 8l — Save pilot extraction comparison to GitHub

Saves per-agent and merged extraction CSVs to the repo and pushes to GitHub.
Run this **after** the pilot (Cell 8i) and again after full extraction (Cell 8j).

In [ ]:
# ── Cell 8l: Save extraction CSVs + push to GitHub ───────────────────────────────
import json, csv, sys, subprocess, os
from pathlib import Path
from datetime import datetime

APP_DIR    = "/content/sysrev/systematic-review"
STATE_FILE = Path(f"{APP_DIR}/data/pipeline_state.jsonl")
OUTPUT_DIR = Path(f"{APP_DIR}/output")
OUTPUT_DIR.mkdir(exist_ok=True)

sys.path.insert(0, APP_DIR)
from pipeline.models import PipelineRecord, PipelineStage

# ── Load all extracted records directly from JSONL state ──────────────────
extracted_records = []
with open(STATE_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        pr = PipelineRecord.model_validate_json(line)
        if pr.pipeline_stage == PipelineStage.EXTRACTION and pr.extracted is not None:
            extracted_records.append(pr)

print(f"Found {len(extracted_records)} extracted records")

if not extracted_records:
    print("No extracted records found — run Cell 8i or 8j first.")
else:
    def _flat(val):
        if isinstance(val, list):
            return " || ".join(str(v) for v in val)
        return val

    rows_a1, rows_a2, rows_merged, rows_compare = [], [], [], []

    for pr in extracted_records:
        ext   = pr.extracted
        title = (pr.dedup.title if pr.dedup else None) or (pr.raw.title if pr.raw else "") or ""
        base  = {
            "record_id": pr.record_id,
            "study_id":  pr.study_id or ext.study_id or "",
            "title":     title,
        }

        if ext.extraction_agent1:
            row = {**base}
            for k, v in ext.extraction_agent1.model_dump().items():
                row[k] = _flat(v)
            rows_a1.append(row)

        if ext.extraction_agent2:
            row = {**base}
            for k, v in ext.extraction_agent2.model_dump().items():
                row[k] = _flat(v)
            rows_a2.append(row)

        if ext.extraction_final:
            row = {**base}
            for k, v in ext.extraction_final.model_dump().items():
                row[k] = _flat(v)
            row["disagreement_fields"] = " || ".join(ext.disagreement_fields)
            row["agents_agree"]        = ext.agents_agree_extraction
            row["human_verified"]      = ext.human_verified
            rows_merged.append(row)

        for field in ext.disagreement_fields:
            a1_val = getattr(ext.extraction_agent1, field, None) if ext.extraction_agent1 else None
            a2_val = getattr(ext.extraction_agent2, field, None) if ext.extraction_agent2 else None
            mg_val = getattr(ext.extraction_final,  field, None) if ext.extraction_final  else None
            rows_compare.append({
                "record_id":    pr.record_id,
                "title":        title[:80],
                "field":        field,
                "agent1":       _flat(a1_val),
                "agent2":       _flat(a2_val),
                "merged":       _flat(mg_val),
                "is_hard_field": field in {
                    "model_name", "workflow_structure", "analytic_task",
                    "human_comparison", "fine_tuned", "rag_used",
                    "formal_methodology", "qualitative_approach"
                }
            })

    def write_csv(path, rows):
        if not rows:
            print(f"  (skipped — no rows) {path.name}")
            return
        with open(path, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=rows[0].keys(), extrasaction="ignore")
            w.writeheader()
            w.writerows(rows)
        print(f"  ✓ {path.name}  ({len(rows)} rows)")

    ts   = datetime.utcnow().strftime("%Y%m%d")
    f_a1   = OUTPUT_DIR / f"extraction_agent1_{ts}.csv"
    f_a2   = OUTPUT_DIR / f"extraction_agent2_{ts}.csv"
    f_mg   = OUTPUT_DIR / f"extraction_merged_{ts}.csv"
    f_diff = OUTPUT_DIR / f"extraction_disagreements_{ts}.csv"

    print(f"\nSaving to {OUTPUT_DIR}/")
    write_csv(f_a1,   rows_a1)
    write_csv(f_a2,   rows_a2)
    write_csv(f_mg,   rows_merged)
    write_csv(f_diff, rows_compare)

    try:
        from google.colab import files as colab_files
        if rows_merged:
            colab_files.download(str(f_mg))
            print(f"\n✓ extraction_merged_{ts}.csv downloaded to your computer")
    except Exception:
        pass

    REPO_DIR = "/content/sysrev"
    print("\nPushing to GitHub…")
    cmds = [
        ["git", "-C", REPO_DIR, "add", "systematic-review/output/"],
        ["git", "-C", REPO_DIR, "commit", "-m",
         f"Add extraction CSVs ({len(rows_merged)} records, {ts})"],
        ["git", "-C", REPO_DIR, "push", "-u", "origin",
         "claude/systematic-review-automation-oC3BZ"],
    ]
    for cmd in cmds:
        result = subprocess.run(cmd, capture_output=True, text=True)
        label  = " ".join(cmd[3:6])
        if result.returncode != 0:
            msg = result.stderr.strip() or result.stdout.strip()
            if "nothing to commit" in msg:
                print(f"  ℹ {label}: nothing new to commit")
            else:
                print(f"  ✗ {label}: {msg[:200]}")
                if "push" in label:
                    break
        else:
            print(f"  ✓ {label}")
    print("\nDone.")


## Cell 9 — Stop the server

In [ ]:
try:
    tunnel.terminate()
    server.terminate()
except Exception:
    pass
os.system('pkill cloudflared 2>/dev/null; true')
print('Stopped.')

---
## Summary

| Database | How searched | Format saved |
|----------|-------------|-------------|
| **PubMed** | Cell 3a (auto, free API) | `pubmed_results.xml` |
| **arXiv** | Cell 3a (auto, free API) | `arxiv_results.json` |
| **OpenAlex** | Cell 3a (auto, free API — covers WoS/Scopus) | `openalex_results.json` |
| **Semantic Scholar** | Cell 3a (auto, free API) | `semanticscholar_results.json` |
| **Scopus** | Cell 3b (manual export, needs login) | `scopus.ris` |
| **Web of Science** | Cell 3b (manual export, needs login) | `wos.ris` |
| **PsycINFO** | Cell 3b (manual export, needs login) | `psycinfo.ris` |
| **ACM DL** | Cell 3b (manual export, needs login) | `acm.bib` |
| **IEEE Xplore** | Cell 3b (manual export, needs login) | `ieee.csv` |

All files land in `data/raw/`. The import stage handles deduplication across sources.
